# Behavioural Plots
---
### author: Juju Fars
### date: 2024-12

Loads the refined patient behavioural data (`Data/Refined/bv_*.xlsx`, produced by
`process_data.ipynb`) and plots pre- vs post-training results for each psychophysics
task: cut-off spatial frequency, contrast sensitivity, interocular imbalance, dichoptic
noise, stereoacuity (asteroid), and crowded/uncrowded acuity (Tumbling E).

In [ ]:
import importlib
import import_packages # all my packages are there, it's easier
importlib.reload(import_packages) # so I can add new packages in import_packages.py without restarting the kernel
from import_packages import *
print("Packages (re)loaded successfully!")

### Set up the paths

In [ ]:
# Set Up the paths 
ori = os.getcwd()

path_data_ref = os.path.abspath(os.path.join(ori, "..", "Data", "Refined"))
path_plot = os.path.abspath(os.path.join(ori, "..", "Data", "Plots"))


path_data_ref_con = os.path.abspath(os.path.join(ori, "..", "Data", "Controls", "Refined"))
path_plot_con = os.path.abspath(os.path.join(ori, "..", "Data", "Controls", "Plots"))

### Load the data

In [ ]:
bv_thresholds = pd.read_excel(path_data_ref+'/bv_2_results_thresholds.xlsx', sheet_name=None) 
bv_diff = pd.read_excel(path_data_ref+'/bv_3_results_Diff.xlsx', sheet_name=None) 

bv_thresholds_con = pd.read_excel(path_data_ref_con+'/bv_2_results_thresholds.xlsx', sheet_name=None) 
bv_diff_con = pd.read_excel(path_data_ref_con+'/bv_3_results_Diff.xlsx', sheet_name=None) 


## Cut-off
Highest spatial frequency (cycles/degree) at which a grating can still be detected -
a measure of acuity.

In [ ]:
import pandas as pd

# cutoff - test
df = bv_diff['cutoff'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID, inplace=True)

df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})

df_long = df.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'],
                  var_name='Stage', value_name='Frequency')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# cutoff - control
df_con = bv_diff_con['cutoff'].copy()
df_con['group'] = 'control'
# MAKE ID A STRING BEFORE ADDING A SUFFIX
# amb mapping (keep consistent / check labels)
df_con['amb'] = df_con['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_con_long = df_con.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'],
                          var_name='Stage', value_name='Frequency')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)

print(df_total)
print(df_long_total)


In [ ]:
df_long_clean = df_long_total.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)
df_long_clean_amb = df_long_clean[(df_long_clean['group'] != 'control')]

In [ ]:


F_cutoff = df_long_clean.copy()
# for the amblyopes
aov = pg.rm_anova(dv='Frequency', within=['amb','Stage'], subject='ID', data=df_long_clean[(df_long_clean['group'] != 'control')])
aov.round(3)

In [ ]:
# for the controls
aov = pg.rm_anova(dv='Frequency', within=['amb','Stage'], subject='ID', data=df_long_clean[(df_long_clean['group'] != 'test')])
aov.round(3)

Must add the singificance values by hand because of the subgroups and the ANOVA

In [ ]:

print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Post-Training']['Frequency'].mean())
print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Post-Training']['Frequency'].std())


In [ ]:
print(df_long_clean[(df_long_clean['group'] != 'test')][df_long_clean['Stage']== 'Pre-Training']['Frequency'].mean())
print(df_long_clean[(df_long_clean['group'] != 'test')][df_long_clean['Stage']== 'Pre-Training']['Frequency'].std())


In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Pre-Training']['Frequency'].values,
    y=df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Post-Training']['Frequency'].values
)
print(ttest.round(3))

### cut-off 

In [ ]:
from matplotlib.lines import Line2D
# Set a better style
sns.set_theme(style="white", palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='Frequency', data=df_long_clean_amb, hue='Stage', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Post-Training': '#009E73', 'Pre-Training': '#D55E00'}
)

ax.axhline(y=22.1, color='#1E90FF', linestyle='--', linewidth=3, zorder=1, alpha=0.35)

strip = sns.stripplot(data=df_long_clean_amb, x="amb", y="Frequency", hue='Stage', dodge=True, ax=ax, 
                palette={'Post-Training': '#009E73', 'Pre-Training': '#D55E00'}, edgecolor='black', linewidth=1)

ax.set_yticks(range(0, 35, 5))
#ax.set_yticklabels([5, 10, 15, 20, 25])

# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles=[(handles[0]), (handles[1])],
#           labels=['Pre-Training', 'Post-training'],
#           loc='lower left', handlelength=3,
#           handler_map={tuple: HandlerTuple(ndivide=None)})

legend_handles = [
    # --- Stage section ---
    # Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color='#D55E00', lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color='#009E73', lw=2.5, label='  Post-Training'),
    Line2D([0], [0], color='#1E90FF', alpha = 0.35, lw=2.5, label='  Control')
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=True,
    fontsize=13,
    handlelength=3
)

# Map amb categories to numerical positions for plotting
amb_order = df_long_clean_amb['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("Frequency (cycles per dva)", fontsize = 14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks
ax.set_title('Pre- and Post-Training Visual Acuity', fontsize = 15)

# Get cleaned categorical order for amb
amb_order = df_long_clean_amb['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'amb' then 'Stage'
df_sorted = df_long_clean_amb.sort_values(by=['amb', 'Stage'], key=lambda col: col.astype(str))

# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, amb) and Stage
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['amb'])
    stage = row['Stage']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][stage] = (x, y)

# Plot lines for each pair of Pre/Post
for key, stages in coord_map.items():
    if 'Pre-Training' in stages and 'Post-training' in stages:
        x_vals = [stages['Pre-Training'][0], stages['Post-training'][0]]
        y_vals = [stages['Pre-Training'][1], stages['Post-training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

# Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 31, 1, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col, fontsize = 15)

ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [28, 28+h, 28+h, 28], lw=1.5, c=col)
ax.text((x1+x1)*.5, 28+h+0.2, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [26, 26+h, 26+h, 26], lw=1.5, c=col)
ax.text((x2+x2)*.5, 26+h+0.2, "n.s.", ha='center', va='bottom', color=col)
#plt.title('No Acuity Improvement for Amblyopes after Training', fontsize = 16)
#plt.figure(figsize=(8, 6))

ax.set_ylim(0, 35)
# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.xticks([0, 1], ['Strong Eye', 'Amblyopic Eye'], fontsize = 13)  # matches 'order=' above

plt.savefig('../Data/Plots/new_new/cutoff.png', dpi=600, bbox_inches='tight')



In [ ]:
mrs_results = pd.read_excel(path_data_ref+'/mrs_results_main.xlsx', sheet_name='metab') 
print(mrs_results.head())
# df_long_clean_amb_eye = df_long_clean_amb[(df_long_clean_amb['amb'] == 'Amblyopic Eye')]

df_long_ccc = df_long_clean_amb.copy()
df_long_ccc = df_long_ccc[(df_long_ccc['Stage'] == 'Post-training')]
print(df_long_ccc.head())

freq_diff = (
    df_long_ccc
    .pivot(index="ID", columns="amb", values="Frequency")
    .assign(freq_diff=lambda x: x["Strong Eye"] - x["Amblyopic Eye"])
    .reset_index()[["ID", "freq_diff"]]
)

In [ ]:
import pandas as pd

df_mrs_long = (
    mrs_results
    .drop(columns="diff")
    .melt(
        id_vars=["ID", "location", "Metab"],
        value_vars=["pre", "post"],
        var_name="Stage",
        value_name="Concentration"
    )
)

df_mrs_long["Stage"] = df_mrs_long["Stage"].map({
    "pre": "Pre-Training",
    "post": "Post-Training"
})


print(df_mrs_long)

In [ ]:
df_mrs_long = df_mrs_long[(df_mrs_long['Metab'] == 'GABA+')]
df_mrs_long = df_mrs_long[(df_mrs_long['location'] == 'V1')]
df_mrs_long = df_mrs_long.drop(columns=["Metab", "location"])

df_mrs_long = df_mrs_long[(df_mrs_long['Stage'] == 'Post-Training')]
df_mrs_long = df_mrs_long.drop(columns=["Stage"])

print(df_mrs_long)


In [ ]:
print(freq_diff)

In [ ]:
df_corr = pd.merge(freq_diff, df_mrs_long, on="ID")


In [ ]:
# check the correlation 

corr = pg.corr(
    y=df_corr['freq_diff'].values,
    x=df_corr['Concentration'].values
)

print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    y=df_corr["freq_diff"],
    x=df_corr["Concentration"],
    alpha=0.7,
    ax=ax
)

ax.set_ylabel("Difference between Strong Eye and Amblyopic Eye\nCut-Off Values (cpd)", loc = 'center')
ax.set_xlabel("Pretraining GABA+ Concentrations (mMol/Kg)")

# ax.text(
#     0.05, 0.95,
#     f"N = 20, R = 0.067\np = 0.778, CI95% = [-0.39, 0.5]",
#     transform=ax.transAxes,
#     ha="left",
#     va="top"
# )
plt.savefig('../Data/Plots/Narrative_plots/v3/Corr_GABA_CUTOFF.png', dpi=600, bbox_inches='tight')

plt.show()

### cut-off full

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================
# DATA
# =========================
df_test = df_long_clean[df_long_clean['group'] == 'test'].copy().reset_index(drop=True)
df_control = df_long_clean[df_long_clean['group'] == 'control'].copy()

# Ensure categorical order for amb
if not pd.api.types.is_categorical_dtype(df_long_clean['amb']):
    amb_order = sorted(df_long_clean['amb'].unique(), key=str)
    df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=amb_order, ordered=True)
    df_test['amb'] = pd.Categorical(df_test['amb'], categories=amb_order, ordered=True)
    df_control['amb'] = pd.Categorical(df_control['amb'], categories=amb_order, ordered=True)
else:
    amb_order = list(df_long_clean['amb'].cat.categories)

# Explicit stage order (important!)
stage_order = ['Pre-Training', 'Post-training']

# =========================
# STYLE
# =========================
sns.set_theme(style="white")

test_palette = {
    'Pre-Training': '#D55E00',
    'Post-training': '#009E73'
}
control_color = '#1E90FF'

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(figsize=(7, 6))

# =========================
# CONTROL BACKGROUND RANGES
# =========================
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'])['Frequency']
    .agg(
        q1=lambda x: x.quantile(0.25),
        median='median',
        q3=lambda x: x.quantile(0.75),
        min='min',
        max='max'
    )
    .reset_index()
)

box_width = 0.6
range_width = 0.36
offsets = {
    'Pre-Training': -box_width / 4,
    'Post-training': box_width / 4
}

for _, row in ctrl_stats.iterrows():
    amb = row['amb']
    stage = row['Stage']
    x_center = amb_order.index(amb) + offsets[stage]

    # min–max range
    ax.add_patch(Rectangle(
        (x_center - range_width / 2, row['min']),
        range_width,
        row['max'] - row['min'],
        color=control_color,
        alpha=0.10,
        zorder=0
    ))

    # IQR
    ax.add_patch(Rectangle(
        (x_center - range_width / 2, row['q1']),
        range_width,
        row['q3'] - row['q1'],
        color=control_color,
        alpha=0.18,
        zorder=0.5
    ))

    # median
    ax.hlines(
        row['median'],
        x_center - range_width / 2,
        x_center + range_width / 2,
        color=control_color,
        linewidth=3,
        zorder=1
    )

# =========================
# TEST BOXPLOTS
# =========================
sns.boxplot(
    data=df_test,
    x='amb',
    y='Frequency',
    hue='Stage',
    order=amb_order,
    hue_order=stage_order,
    width=box_width,
    palette=test_palette,
    linewidth=1.5,
    showfliers=False,
    ax=ax,
    zorder=2
)

# =========================
# TEST DOTS (MANUAL)
# =========================
jitter = 0.06
rng = np.random.default_rng(42)

# --- compute x positions for manual scatter (fixed dtype)
df_test['x_base'] = df_test['amb'].map({a: i for i, a in enumerate(amb_order)}).astype(float)
df_test['x_offset'] = df_test['Stage'].map(offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']


for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['Frequency'],
        color=test_palette[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=4,
        label=stage
    )

# =========================
# CONNECT PRE ↔ POST (TEST)
# =========================
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]

        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['Frequency'], post['Frequency']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=3
        )

# =========================
# AXES / LEGEND / DECOR
# =========================
control_median_all = df_control['Frequency'].median()


ax.axhline(
    y=control_median_all,
    color=control_color,      # same blue as control shading
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)
ax.set_ylabel("Frequency (cycles per dva)")
ax.set_xlabel("")
ax.set_yticks(range(0, 35, 5))
ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)


x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 32, 1, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)

ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [28, 28+h, 28+h, 28], lw=1.5, c=col)
ax.text((x1+x1)*.5, 28+h+0.2, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [26, 26+h, 26+h, 26], lw=1.5, c=col)
ax.text((x2+x2)*.5, 26+h+0.2, "n.s.", ha='center', va='bottom', color=col)

# clean legend (avoid duplicates)
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))
ax.legend(
    [uniq['Pre-Training'], uniq['Post-training']],
    ['Pre-Training', 'Post-training'],
    loc='lower left',
    handlelength=3
)

plt.tight_layout()
plt.savefig(
    '../Data/Plots/Narrative_plots/cutoff.png',
    dpi=600,
    bbox_inches='tight'
)
plt.show()


In [ ]:

# Set a better style
sns.set_theme(style="whitegrid", palette="pastel")

# Set up the plot
plt.figure(figsize=(4, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='Frequency', data=df_long_clean_BE, hue='Stage', showfliers=False, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'}
)

ax.axhline(y=22.1, color='blue', linestyle='--', linewidth=3, zorder=1, alpha=0.2)

strip = sns.stripplot(data=df_long_clean_BE, x="amb", y="Frequency", hue='Stage', dodge=True, ax=ax, 
                palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'}, edgecolor='black', linewidth=1)

ax.set_yticks(range(0, 35, 5))
ax.set_ylim(10, 30)
#ax.set_yticklabels([5, 10, 15, 20, 25])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1])],
          labels=['Pre-Training', 'Post-training'],
          loc='lower right', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("Frequency (cycles per dva)")  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean_BE['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'amb' then 'Stage'
df_sorted = df_long_clean_BE.sort_values(by=['amb', 'Stage'], key=lambda col: col.astype(str))

# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, amb) and Stage
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['amb'])
    stage = row['Stage']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][stage] = (x, y)

# Plot lines for each pair of Pre/Post
for key, stages in coord_map.items():
    if 'Pre-Training' in stages and 'Post-training' in stages:
        x_vals = [stages['Pre-Training'][0], stages['Post-training'][0]]
        y_vals = [stages['Pre-Training'][1], stages['Post-training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

# # Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 27, 0.5, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [28, 28+h, 28+h, 28], lw=1.5, c=col)
ax.text((x1+x1)*.5, 28+h+0.2, "n.s.", ha='center', va='bottom', color=col)

#plt.title('No Acuity Improvement for Amblyopes after Training', fontsize = 16)
#plt.figure(figsize=(8, 6))
plt.savefig('../Data/Plots/Narrative_plots/cutoff_supplementary.png', dpi=600, bbox_inches='tight')



### cut-off - both eyes

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================
# DATA
# =========================
df_test = df_long_clean_BE[df_long_clean_BE['group'] == 'test'].copy().reset_index(drop=True)
df_control = df_long_clean_BE[df_long_clean_BE['group'] == 'control'].copy()

# Ensure categorical order for amb
if not pd.api.types.is_categorical_dtype(df_long_clean_BE['amb']):
    amb_order = sorted(df_long_clean_BE['amb'].unique(), key=str)
    df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=amb_order, ordered=True)
    df_test['amb'] = pd.Categorical(df_test['amb'], categories=amb_order, ordered=True)
    df_control['amb'] = pd.Categorical(df_control['amb'], categories=amb_order, ordered=True)
else:
    amb_order = list(df_long_clean_BE['amb'].cat.categories)

# Explicit stage order (important!)
stage_order = ['Pre-Training', 'Post-training']

# =========================
# STYLE
# =========================
sns.set_theme(style="white")

test_palette = {
    'Pre-Training': '#D55E00',
    'Post-training': '#009E73'
}
control_color = '#1E90FF'

# =========================
# FIGURE
# =========================
fig, ax = plt.subplots(figsize=(4, 6))

# =========================
# CONTROL BACKGROUND RANGES
# =========================
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'])['Frequency']
    .agg(
        q1=lambda x: x.quantile(0.25),
        median='median',
        q3=lambda x: x.quantile(0.75),
        min='min',
        max='max'
    )
    .reset_index()
)

box_width = 0.4
range_width = 0.36
offsets = {
    'Pre-Training': -box_width / 4,
    'Post-training': box_width / 4
}

for _, row in ctrl_stats.iterrows():
    amb = row['amb']
    stage = row['Stage']
    x_center = amb_order.index(amb) + offsets[stage]

    # min–max range
    ax.add_patch(Rectangle(
        (x_center - range_width / 2, row['min']),
        range_width,
        row['max'] - row['min'],
        color=control_color,
        alpha=0.10,
        zorder=0
    ))

    # IQR
    ax.add_patch(Rectangle(
        (x_center - range_width / 2, row['q1']),
        range_width,
        row['q3'] - row['q1'],
        color=control_color,
        alpha=0.18,
        zorder=0.5
    ))

    # median
    ax.hlines(
        row['median'],
        x_center - range_width / 2,
        x_center + range_width / 2,
        color=control_color,
        linewidth=3,
        zorder=1
    )

# =========================
# TEST BOXPLOTS
# =========================
sns.boxplot(
    data=df_test,
    x='amb',
    y='Frequency',
    hue='Stage',
    order=amb_order,
    hue_order=stage_order,
    width=box_width,
    palette=test_palette,
    linewidth=1.5,
    showfliers=False,
    ax=ax,
    zorder=2
)

# =========================
# TEST DOTS (MANUAL)
# =========================
jitter = 0.06
rng = np.random.default_rng(42)

# --- compute x positions for manual scatter (fixed dtype)
df_test['x_base'] = df_test['amb'].map({a: i for i, a in enumerate(amb_order)}).astype(float)
df_test['x_offset'] = df_test['Stage'].map(offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']


for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['Frequency'],
        color=test_palette[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=4,
        label=stage
    )

# =========================
# CONNECT PRE ↔ POST (TEST)
# =========================
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]

        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['Frequency'], post['Frequency']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=3
        )

# =========================
# AXES / LEGEND / DECOR
# =========================
control_median_all = df_control['Frequency'].median()


ax.axhline(
    y=control_median_all,
    color=control_color,      # same blue as control shading
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)
ax.set_ylabel("Frequency (cycles per dva)")
ax.set_xlabel("")
ax.set_yticks(range(10, 30, 5))
ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)

# clean legend (avoid duplicates)
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))
ax.legend(
    [uniq['Pre-Training'], uniq['Post-training']],
    ['Pre-Training', 'Post-training'],
    loc='lower left',
    handlelength=3
)

plt.tight_layout()
# plt.savefig(
#     '../Data/Plots/Narrative_plots/cutoff.png',
#     dpi=600,
#     bbox_inches='tight'
# )
plt.show()


In [ ]:
# Run paired for the controls
ttest = pg.ttest(
    paired=True,
    x=df_long_clean_BE[(df_long_clean_BE['group'] != 'test')][df_long_clean_BE['Stage'] == 'Pre-Training']['Frequency'].values,
    y=df_long_clean_BE[(df_long_clean_BE['group'] != 'test')][df_long_clean_BE['Stage'] == 'Post-training']['Frequency'].values
)

print(ttest.round(3))

In [ ]:
# Run paired for the test


ttest = pg.ttest(
    paired=True,
    x=df_long_clean_BE[(df_long_clean_BE['group'] != 'control')][df_long_clean_BE['Stage'] == 'Pre-Training']['Frequency'].values,
    y=df_long_clean_BE[(df_long_clean_BE['group'] != 'control')][df_long_clean_BE['Stage'] == 'Post-training']['Frequency'].values
)

print(ttest.round(3))

In [ ]:
df_amb = df.copy()
df_amb = df_amb[(df_amb['amb'] == 'Lazy Eye')]

In [ ]:

offsets = {
    'P20': (3, -0.2),
    'P15': (0.5, -2),
    'P16': (2, 0.5),
    'P07':  (3, -0.3),
}

fig, ax = plt.subplots(figsize=(4, 4))

for _, row in df_amb.iterrows():
    x = row['pretraining']
    y = row['posttraining']
    id_ = row['ID']

    if id_ in offsets:
        dx, dy = offsets[id_]

        # draw a small dot at the true data point only for shifted labels
        ax.plot(x, y, 'ko', markersize=3, zorder=3)

        ha = 'left' if dx > 0 else ('right' if dx < 0 else 'center')
        va = 'bottom' if dy > 0 else ('top' if dy < 0 else 'center')

        ax.annotate(
            str(id_),
            xy=(x, y),
            xytext=(x + dx, y + dy),
            fontsize=14,
            ha=ha, va=va,
            xycoords='data',
            textcoords='data',
            arrowprops=dict(
                arrowstyle='-',
                linewidth=1.0,
                color='black',
                shrinkA=6,   # shortens line so it doesn't go under the text
                shrinkB=0
            ),
            zorder=4
        )
    else:
        # regular labels: no dot, no connector
        ax.text(x, y, str(id_), fontsize=14, ha='center', va='center', zorder=4)

# reference line y = x
xmin = df_amb['pretraining'].min() - 2
xmax = df_amb['pretraining'].max() + 3
ax.plot([xmin, xmax], [xmin, xmax], '--', color='gray')

ax.set_xlim(5, 27)
ax.set_ylim(5, 27)
ax.set_xticks(range(5, 30, 5))
ax.set_yticks(range(5, 30, 5))
ax.set_xlabel("Pre-training Frequency (cpd)")
ax.set_ylabel("Post-training Frequency (cpd)")
ax.set_title("Lazy Eye Training (pre- vs post- training)")
ax.grid(True)

plt.tight_layout()
plt.savefig('../Data/Plots/Narrative_plots/cutoff_pre_vs_post_amb.png', dpi=600, bbox_inches='tight')

# plt.show()


## Contrast sensitivity

This part of the script will analyse the contrast sensitivity results. First the data extraction. Then the descriptive analysis: thresholds, normalised CSF curves. And finally the statistics   

In [ ]:
# for the test
df = bv_diff['contrast_sensitivity'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'Staircase', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='Contrast')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

#for the control
df_con = bv_diff_con['contrast_sensitivity'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_con_long = df_con.melt(id_vars=['ID', 'Staircase', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='Contrast')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
print(df_total)
print(df_long_total)

In [ ]:
df_long_clean = df_long_total.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)
df_long_clean_BE['Sensitivity'] = df_long_clean_BE['Contrast']

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)
df_long_clean['Sensitivity'] = df_long_clean['Contrast']
print(df_long_clean.head())

In [ ]:
# Define the tick labels (assuming Staircase is 1 to 6)
sf_labels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
df_long_clean[(df_long_clean['group'] != 'control')]['Staircase'] = df_long_clean[(df_long_clean['group'] != 'control')]['Staircase'].astype(int)  # ensure int for sorting

# Set theme
sns.set_theme(style="whitegrid", palette="pastel")

# Define facet order
amb_order = ['Strong Eye', 'Amblyopic Eye']  # adjust this to your actual category names

# Create facet grid with vertical arrangement
g = sns.catplot(
    data=df_long_clean[(df_long_clean['group'] != 'control')], kind="box",
    x="Staircase", y="Contrast",
    hue="Stage", row="amb", row_order=amb_order,
    showfliers=False, width=0.6, linewidth=1.5,
    palette={'Post-Training': '#009E73', 'Pre-Training': '#D55E00'},
    height=4, aspect=1.4, sharex=True, sharey=True
)

# Overlay stripplot
for ax, (amb_label, sub_df) in zip(g.axes.flat, df_long_clean[(df_long_clean['group'] != 'control')].groupby("amb")):
    sns.stripplot(
        data=sub_df, x="Staircase", y="Contrast", hue="Stage",
        dodge=True, ax=ax, edgecolor='black', linewidth=1,
        palette={'Post-Training': '#009E73', 'Pre-Training': '#D55E00'},
        legend=False
    )
    ax.set_yscale('log')  # Set log scale for y-axis
    ax.set_yticks([1, 10, 100])
    ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())  # show full values, not scientific

    ax.set_xticks(np.arange(6))  # Set the ticks
    ax.set_xticklabels(sf_labels)  # Set the frequency labels
    ax.text(
    0.45, 1.05, amb_label, transform=ax.transAxes,
    fontsize=12, fontweight='bold', verticalalignment='top',
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black")
)

# Final touches
g.set_axis_labels("Spatial Frequency - Cut-Off threshold ratio", None)  # Remove individual y-labels

# Remove individual y-axis labels
for ax in g.axes.flat:
    ax.set_ylabel('')

# Add shared y-axis label for the entire figure
g.fig.text(
    0.04, 0.5, 'Sensitivity (1/Contrast of % Weber)',
    va='center', rotation='vertical', fontsize=12
)
#g.set_titles(row_template="{row_name}")
g.set_titles('')
# Add title inside a box

#g.add_legend(title="Stage")

# Remove automatic legend
g._legend.remove()

# Get the first axis (top plot)
first_ax = g.axes[0][0]

# Manually create legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#D55E00', edgecolor='black', label='Pre-Training'),
    Patch(facecolor='#009E73', edgecolor='black', label='Post-Training')
]

# Add custom legend to the bottom left
first_ax.legend(
    handles=legend_elements,
    title='Stage',
    loc='lower left',  # places it inside the plot
    frameon=True,
    fontsize=10,
    title_fontsize=11
)

g.savefig('../Data/Plots/Narrative_plots/contrast_curves.png', dpi=600, bbox_inches='tight')


In [ ]:
df_temp = df_long_clean[(df_long_clean['group'] != 'control')]
print(df_long_clean)

### contrast sensitivity curves 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D

# -------------------------
# Prepare data
# -------------------------
# use df_temp as provided by you; filter to test group only
df = df_long_clean.copy()
df = df[df['group'] == 'test'].reset_index(drop=True)

# map staircase -> spatial frequency
sf_map = {
    1: 1/16,
    2: 1/8,
    3: 1/4,
    4: 1/2,
    5: 3/4,
    6: 1
}
df['SF'] = df['Staircase'].map(sf_map)

# make sure amb values match expected strings; list unique to check if needed:
# print(df['amb'].unique())

# -------------------------
# Aggregate (means + sem)
# -------------------------
agg = (
    df
    .groupby(['amb', 'Stage', 'SF'], as_index=False)
    .agg(
        mean_cs=('Sensitivity', 'mean'),
        sem_cs=('Sensitivity', 'sem')
    )
)

# -------------------------
# Fit log-parabola in log10 space
# -------------------------

fits = {}
for (amb, stage), sub in agg.groupby(['amb', 'Stage']):
    x = sub['SF'].values
    y = np.log10(sub['mean_cs'].values)

    # initial guesses: a ~ max(y), b small positive, f_peak ~ SF at max mean
    if len(x) < 3:
        fits[(amb, stage)] = None
        continue

    p0 = [np.max(y), 0.8, x[np.argmax(y)]]
    try:
        popt, _ = curve_fit(log_parabola, x, y, p0=p0, maxfev=10000)
        fits[(amb, stage)] = popt
    except Exception:
        fits[(amb, stage)] = None

# -------------------------
# Plotting: both amb on same axes
# -------------------------
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))

# desired orders and colors
amb_order = ['Strong Eye', 'Amblyopic Eye']  # adjust if your labels differ
stage_order = ['Pre-Training', 'Post-Training']
colors = {'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
linestyles = {'Strong Eye': '-', 'Amblyopic Eye': '--'}

# dense SFs for smooth fitted curves
sf_dense = np.logspace(np.log10(1/16), np.log10(1), 400)

# Plot raw points (low alpha) for test only, but color by Stage and shape by amb (optional)
for stage in stage_order:
    sub = df[df['Stage'] == stage]
    ax.scatter(
        sub['SF'],
        sub['Sensitivity'],
        color=colors[stage],
        alpha=0.12,
        s=18,
        zorder=1,
        label=None if stage == 'Pre-Training' else None  # avoid legend duplication here
    )

# Plot aggregated means+SEM and fitted curves for each amb × stage
for amb in amb_order:
    sub_agg = agg[agg['amb'] == amb]
    for stage in stage_order:
        stage_df = sub_agg[sub_agg['Stage'] == stage].sort_values('SF')
        if stage_df.empty:
            continue

        # mean + sem points (markers)
        ax.errorbar(
            stage_df['SF'],
            stage_df['mean_cs'],
            yerr=stage_df['sem_cs'],
            fmt='o',
            color=colors[stage],
            capsize=3,
            zorder=3,
            label=f"{amb} — {stage}"  # label each curve separately (4 labels)
        )

        # fitted curve (if available)
        popt = fits.get((amb, stage))
        if popt is not None:
            y_fit = 10**bv.log_parabola(sf_dense, *popt)
            ax.plot(
                sf_dense,
                y_fit,
                color=colors[stage],
                linestyle=linestyles.get(amb, '-'),
                linewidth=2.5,
                zorder=4
            )

# -------------------------
# Formatting axes (log-log)
# -------------------------
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlim(0.055, 1.125)
ax.set_ylim(1, 100)

sf_ticks = np.array([1/16, 1/8, 1/4, 1/2, 3/4, 1])
sf_ticklabels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
ax.set_xticks(sf_ticks)
ax.set_xticklabels(sf_ticklabels)

ax.set_yticks([1, 10, 100])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# Labels
ax.set_xlabel('Spatial Frequency (Staircase - Ratio to Cut-Off Threshold)')
ax.set_ylabel('Contrast Sensitivity (1 / Weber Contrast)')

# Legend: it will contain four entries (amb × stage)
ax.legend(title='Eye — Stage', fontsize=9, title_fontsize=10, frameon=True, loc='lower left')



legend_handles = [
    # --- Stage section ---
    Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training'),

    # spacer
    Line2D([], [], linestyle='none', label=''),

    # --- Eye section ---
    Line2D([], [], linestyle='none', label='Eye'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='-', label='  Strong Eye'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='  Amblyopic Eye'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=True,
    fontsize=10,
    handlelength=3
)


plt.tight_layout()
# plt.savefig('../Data/Plots/Narrative_plots/cs_curves_test_only.png',
            # dpi=600, bbox_inches='tight')
plt.show()


### Contrast - full

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D

# -------------------------
# Data setup (use df_temp)
# -------------------------
df_all = df_long_clean.copy()

# map staircase -> spatial frequency
sf_map = {1: 1/16, 2: 1/8, 3: 1/4, 4: 1/2, 5: 3/4, 6: 1}
df_all['SF'] = df_all['Staircase'].map(sf_map)

# split test vs control
df_test = df_all[df_all['group'] == 'test'].reset_index(drop=True)
df_control = df_all[df_all['group'] == 'control'].reset_index(drop=True)

# orders / styles
amb_order = ['Strong Eye', 'Amblyopic Eye']   # adjust if needed
stage_order = ['Pre-Training', 'Post-Training']
colors = {'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
linestyles = {'Strong Eye': '-', 'Amblyopic Eye': '--'}
control_color = '#1E90FF'

# -------------------------
# Aggregate test (means + sem) -- unchanged
# -------------------------
agg_test = (
    df_test
    .groupby(['amb', 'Stage', 'SF'], as_index=False)
    .agg(mean_cs=('Sensitivity', 'mean'), sem_cs=('Sensitivity', 'sem'))
)

# -------------------------
# Compute pooled control stats per SF for envelope + mean for fitting
# -------------------------
ctrl_pooled = (
    df_control
    .groupby('SF', as_index=False)
    .agg(
        minv=('Sensitivity', 'min'),
        q1=('Sensitivity', lambda x: x.quantile(0.25)),
        median=('Sensitivity', 'median'),
        q3=('Sensitivity', lambda x: x.quantile(0.75)),
        maxv=('Sensitivity', 'max'),
        mean_cs=('Sensitivity', 'mean'),
        sem_cs=('Sensitivity', 'sem'),
        n=('Sensitivity', 'count')
    )
    .sort_values('SF')
)

# -------------------------
# Fit log-parabola to test aggregated means (existing)
# -------------------------
fits = {}
for (amb, stage), sub in agg_test.groupby(['amb', 'Stage']):
    x = sub['SF'].values
    y = np.log10(sub['mean_cs'].values)
    if len(x) < 3:
        fits[(amb, stage)] = None
        continue
    p0 = [np.max(y), 0.8, x[np.argmax(y)]]
    try:
        popt, _ = curve_fit(log_parabola, x, y, p0=p0, maxfev=10000)
        fits[(amb, stage)] = popt
    except Exception:
        fits[(amb, stage)] = None

# -------------------------
# Fit single control curve to pooled control mean_cs (same model)
# -------------------------
control_fit = None
if not ctrl_pooled.empty and ctrl_pooled['n'].sum() > 0:
    x_ctrl = ctrl_pooled['SF'].values
    y_ctrl = np.log10(ctrl_pooled['mean_cs'].values)
    if len(x_ctrl) >= 3:
        p0 = [np.max(y_ctrl), 0.8, x_ctrl[np.argmax(y_ctrl)]]
        try:
            control_fit, _ = curve_fit(log_parabola, x_ctrl, y_ctrl, p0=p0, maxfev=10000)
        except Exception:
            control_fit = None

# -------------------------
# Plotting
# -------------------------
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))
sf_dense = np.logspace(np.log10(1/16), np.log10(1), 400)

# ---- Plot the fitted control curve (single curve), similar styling to test fits ----
if control_fit is not None:
    y_fit_ctrl = 10**bv.log_parabola(sf_dense, *control_fit)
    ax.plot(
        sf_dense,
        y_fit_ctrl,
        color=control_color,
        lw=2.8,
        alpha=0.5,
        zorder=1.5,
        linestyle='-',
        label='Control (fitted)'
    )

# ---- Raw test points (faint) ----
for stage in stage_order:
    s = df_test[df_test['Stage'] == stage]
    ax.scatter(
        s['SF'],
        s['Sensitivity'],
        color=colors[stage],
        alpha=0.12,
        s=18,
        zorder=2
    )

# ---- Plot aggregated means + sem + fitted curves for test (4 curves) ----
for amb in amb_order:
    sub_agg = agg_test[agg_test['amb'] == amb]
    for stage in stage_order:
        stage_df = sub_agg[sub_agg['Stage'] == stage].sort_values('SF')
        if stage_df.empty:
            continue

        ax.errorbar(
            stage_df['SF'],
            stage_df['mean_cs'],
            yerr=stage_df['sem_cs'],
            fmt='o',
            color=colors[stage],
            capsize=3,
            zorder=4,
            label=None
        )

        popt = fits.get((amb, stage))
        if popt is not None:
            y_fit = 10**bv.log_parabola(sf_dense, *popt)
            ax.plot(
                sf_dense,
                y_fit,
                color=colors[stage],
                linestyle=linestyles.get(amb, '-'),
                linewidth=2.6,
                zorder=5,
                label=None
            )

# ---- Axes formatting ----
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.055, 1.125)
ax.set_ylim(1, 100)

sf_ticks = np.array([1/16, 1/8, 1/4, 1/2, 3/4, 1])
sf_ticklabels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
ax.set_xticks(sf_ticks)
ax.set_xticklabels(sf_ticklabels)
ax.set_yticks([1, 10, 100])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax.set_xlabel('Spatial Frequency (Staircase - Ratio to Cut-Off Threshold)', fontsize = 14)
ax.set_ylabel('Contrast Sensitivity (1 / Weber Contrast)', fontsize = 14)
ax.set_title('Contrast Sensitivity Curves', fontsize = 15)
# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)

# ---- Combined legend (one box): Stage (colour) + Eye (linestyle) and a control entry ----
legend_handles = [
    Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training'),
    Line2D([], [], linestyle='none', label=''),
    Line2D([], [], linestyle='none', label='Eye'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='-', label='  Strong Eye'),
    Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='  Amblyopic Eye'),
    Line2D([], [], linestyle='none', label=''),
    Line2D([0], [0], color=control_color, lw=2.5, alpha = 0.5, label='Control')
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, fontsize=12, handlelength=3)

plt.tight_layout()
plt.savefig('../Data/Plots/new_new/cs_curves_control_fitted_envelope.png',
            dpi=600, bbox_inches='tight')
plt.show()


### contrast - both eyes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D

# -------------------------
# Prepare data
# -------------------------
# use df_temp as provided by you; filter to test group only
df = df_long_clean_BE.copy()
df = df[df['group'] == 'test'].reset_index(drop=True)

# map staircase -> spatial frequency
sf_map = {
    1: 1/16,
    2: 1/8,
    3: 1/4,
    4: 1/2,
    5: 3/4,
    6: 1
}
df['SF'] = df['Staircase'].map(sf_map)

# make sure amb values match expected strings; list unique to check if needed:
# print(df['amb'].unique())

# -------------------------
# Aggregate (means + sem)
# -------------------------
agg = (
    df
    .groupby(['amb', 'Stage', 'SF'], as_index=False)
    .agg(
        mean_cs=('Sensitivity', 'mean'),
        sem_cs=('Sensitivity', 'sem')
    )
)

# -------------------------
# Fit log-parabola in log10 space
# -------------------------

fits = {}
for (amb, stage), sub in agg.groupby(['amb', 'Stage']):
    x = sub['SF'].values
    y = np.log10(sub['mean_cs'].values)

    # initial guesses: a ~ max(y), b small positive, f_peak ~ SF at max mean
    if len(x) < 3:
        fits[(amb, stage)] = None
        continue

    p0 = [np.max(y), 0.8, x[np.argmax(y)]]
    try:
        popt, _ = curve_fit(log_parabola, x, y, p0=p0, maxfev=10000)
        fits[(amb, stage)] = popt
    except Exception:
        fits[(amb, stage)] = None

# -------------------------
# Plotting: both amb on same axes
# -------------------------
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))

# desired orders and colors
amb_order = ['Both Eyes']  # adjust if your labels differ
stage_order = ['Pre-Training', 'Post-training']
colors = {'Pre-Training': '#D55E00', 'Post-training': '#009E73'}
linestyles = {'Both Eyes': '-'}

# dense SFs for smooth fitted curves
sf_dense = np.logspace(np.log10(1/16), np.log10(1), 400)

# Plot raw points (low alpha) for test only, but color by Stage and shape by amb (optional)
for stage in stage_order:
    sub = df[df['Stage'] == stage]
    ax.scatter(
        sub['SF'],
        sub['Sensitivity'],
        color=colors[stage],
        alpha=0.12,
        s=18,
        zorder=1,
        label=None if stage == 'Pre-Training' else None  # avoid legend duplication here
    )

# Plot aggregated means+SEM and fitted curves for each amb × stage
for amb in amb_order:
    sub_agg = agg[agg['amb'] == amb]
    for stage in stage_order:
        stage_df = sub_agg[sub_agg['Stage'] == stage].sort_values('SF')
        if stage_df.empty:
            continue

        # mean + sem points (markers)
        ax.errorbar(
            stage_df['SF'],
            stage_df['mean_cs'],
            yerr=stage_df['sem_cs'],
            fmt='o',
            color=colors[stage],
            capsize=3,
            zorder=3,
            label=f"{amb} — {stage}"  # label each curve separately (4 labels)
        )

        # fitted curve (if available)
        popt = fits.get((amb, stage))
        if popt is not None:
            y_fit = 10**bv.log_parabola(sf_dense, *popt)
            ax.plot(
                sf_dense,
                y_fit,
                color=colors[stage],
                linestyle=linestyles.get(amb, '-'),
                linewidth=2.5,
                zorder=4
            )

# -------------------------
# Formatting axes (log-log)
# -------------------------
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlim(0.055, 1.125)
ax.set_ylim(1, 100)

sf_ticks = np.array([1/16, 1/8, 1/4, 1/2, 3/4, 1])
sf_ticklabels = ['1/16', '1/8', '1/4', '1/2', '3/4', '1']
ax.set_xticks(sf_ticks)
ax.set_xticklabels(sf_ticklabels)

ax.set_yticks([1, 10, 100])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# Labels
ax.set_xlabel('Spatial Frequency (Staircase - Ratio to Cut-Off Threshold)')
ax.set_ylabel('Contrast Sensitivity (1 / Weber Contrast)')

# Legend: it will contain four entries (amb × stage)
ax.legend(title='Both Eyes — Stage', fontsize=9, title_fontsize=10, frameon=True, loc='lower left')



legend_handles = [
    # --- Stage section ---
    Line2D([], [], linestyle='none', label='Both Eyes - Stage'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-training'], lw=2.5, label='  Post-training'),

    # # spacer
    # Line2D([], [], linestyle='none', label=''),

    # # --- Eye section ---
    # Line2D([], [], linestyle='none', label='Eye'),
    # Line2D([0], [0], color='black', lw=2.5, linestyle='-', label='  Strong Eye'),
    # Line2D([0], [0], color='black', lw=2.5, linestyle='--', label='  Amblyopic Eye'),
]

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=True,
    fontsize=10,
    handlelength=3
)


plt.tight_layout()
# plt.savefig('../Data/Plots/Narrative_plots/cs_curves_test_only.png',
            # dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import pingouin as pg

# Make sure your dataframe is called df and columns match exactly:
# 'ID', 'Staircase', 'Stage', 'logBP'
# df_long_clean[(df_long_clean['group'] != 'control')]
# Convert factors to categorical
df_long_clean['ID'] = df_long_clean['ID'].astype('category')
df_long_clean['Stage'] = df_long_clean['Stage'].astype('category')
df_long_clean['Staircase'] = df_long_clean['Staircase'].astype('category')

# Run two-way repeated measures ANOVA for test 
aov = pg.rm_anova(
    dv='Sensitivity',
    within=['Stage', 'Staircase'],
    subject='ID',
    data=df_long_clean[(df_long_clean['group'] != 'control')],
    detailed=True
)

print(aov)

In [ ]:
# Run two-way repeated measures ANOVA for controls
aov = pg.rm_anova(
    dv='Sensitivity',
    within=['Stage', 'Staircase'],
    subject='ID',
    data=df_long_clean[(df_long_clean['group'] != 'test')],
    detailed=True
)

print(aov)

### contrast - AUC

In [ ]:
bv_auc = pd.read_excel(path_data_ref+'/bv_4_results_ABC_diff.xlsx', sheet_name=None) 
bv_auc_con = pd.read_excel(path_data_ref_con+'/bv_4_results_ABC_diff.xlsx', sheet_name=None)


In [ ]:
# for the test
df = bv_auc['abc_df_in'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='ABC_full')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# for the controls
df_con = bv_auc_con['abc_df_in'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_con_long = df_con.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='ABC_full')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
print(df_total)
print(df_long_total)

In [ ]:
df_long_clean = df_long_total.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)
print(df_long_clean.head())

In [ ]:
print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Strong Eye']['ABC_full'].mean())
print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Strong Eye']['ABC_full'].std())

print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye']['ABC_full'].mean())
print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye']['ABC_full'].std())

print(df_long_clean[(df_long_clean['group'] == 'control')]['ABC_full'].mean())
print(df_long_clean[(df_long_clean['group'] == 'control')]['ABC_full'].std())

print(df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Pre-Training']['ABC_full'].mean())


In [ ]:
F_contrast = df_long_clean.copy()

In [ ]:
aov = pg.rm_anova(dv='ABC_full', within=['amb','Stage'], subject='ID', data=df_long_clean[(df_long_clean['group'] != 'control')])
aov.round(3)

In [ ]:
aov = pg.rm_anova(dv='ABC_full', within=['amb','Stage'], subject='ID', data=df_long_clean[(df_long_clean['group'] != 'test')])
aov.round(3)

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Pre-Training']['ABC_full'].values,
    y=df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Post-training']['ABC_full'].values
)
print(ttest.round(3))

In [ ]:

# Set a better style
sns.set_theme(style="whitegrid", palette="pastel")

# Set up the plot
plt.figure(figsize=(7, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='ABC_full', data=df_long_clean, hue='Stage', showfliers=False, width=0.6, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Post-Training': '#009E73', 'Pre-Training': '#D55E00'}
)

# ax.axhline(y=0.5, color='blue', linestyle='--', linewidth=3, zorder=1)


strip = sns.stripplot(data=df_long_clean, x="amb", y="ABC_full", hue='Stage', dodge=True, ax=ax, 
                palette={'Post-Training': '#009E73', 'Pre-Training': '#D55E00'}, edgecolor='black', linewidth=1)

ax.set_yticks(range(0, 3000, 250))
#ax.set_yticklabels([5, 10, 15, 20, 25])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1])],
          labels=['Pre-Training', 'Post-Training'],
          loc='lower left', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("Area Under the Curve (AUC)")  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'amb' then 'Stage'
df_sorted = df_long_clean.sort_values(by=['amb', 'Stage'], key=lambda col: col.astype(str))

# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, amb) and Stage
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['amb'])
    stage = row['Stage']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][stage] = (x, y)

# Plot lines for each pair of Pre/Post
for key, stages in coord_map.items():
    if 'Pre-Training' in stages and 'Post-Training' in stages:
        x_vals = [stages['Pre-Training'][0], stages['Post-Training'][0]]
        y_vals = [stages['Pre-Training'][1], stages['Post-Training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

# Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 3000, 50, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)
h2 = 2750
ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x1+x1)*.5, h2+h+20, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x2+x2)*.5, h2+h+20, "n.s.", ha='center', va='bottom', color=col)
# plt.title('No Contrast Sensitivity Improvement in Amblyopes', fontsize=18)
# plt.figure(figsize=(8, 6))

plt.savefig('../Data/Plots/Narrative_plots/contrast_AUC.png', dpi=600, bbox_inches='tight')


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# -----------------------
# CONFIG / pick y column
# -----------------------
y_col = 'ABC_full' if 'ABC_full' in df_long_clean.columns else ('ABC' if 'ABC' in df_long_clean.columns else None)
if y_col is None:
    raise ValueError("Couldn't find 'ABC_full' or 'ABC' in df_long_clean columns.")

box_width = 0.6
range_width = 0.36   # width (in x data units) for the control vertical boxes
jitter = 0.06
rng = np.random.default_rng(42)

# colours / styles
colors = {'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
control_color = '#1E90FF'
linestyles = {'Strong Eye': '-', 'Amblyopic Eye': '--'}

# -----------------------
# Prepare dataframes
# -----------------------
df_all = df_long_clean.copy()

# ensure ID is string so grouping is stable
df_all['ID'] = df_all['ID'].astype(str)

# ensure Stage names are exactly as we expect
# if there are small differences (e.g., 'Post-training' vs 'Posttraining') this won't fix them,
# but it's easy to adapt if needed.
stage_order = ['Pre-Training', 'Post-Training']

# amb order from categorical if available, else infer
if pd.api.types.is_categorical_dtype(df_all['amb']):
    amb_order = list(df_all['amb'].cat.categories)
else:
    amb_order = sorted(df_all['amb'].unique(), key=str)
    df_all['amb'] = pd.Categorical(df_all['amb'], categories=amb_order, ordered=True)

# split test vs control
df_test = df_all[df_all['group'] == 'test'].reset_index(drop=True)
df_control = df_all[df_all['group'] == 'control'].reset_index(drop=True)

# -----------------------
# Control stats (per amb × Stage × SF equivalent)
# We'll compute per-(amb,Stage) min, q1, median, q3, max of y_col
# -----------------------
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'], as_index=False)[y_col]
    .agg(q1=lambda x: x.quantile(0.25),
         median='median',
         q3=lambda x: x.quantile(0.75),
         minv='min',
         maxv='max',
         n='count')
)

# -----------------------
# Make figure
# -----------------------
sns.set_theme(style="white", palette="pastel")
fig, ax = plt.subplots(figsize=(7, 6))

# draw control vertical ranges behind boxes
for _, row in ctrl_stats.iterrows():
    amb = row['amb']
    stage = row['Stage']
    # x center = index of amb + small offset depending on stage
    x_center = amb_order.index(amb)
    # offsets for pre/post (match how boxplot/hue dodge will place them)
    offset = -box_width/4 if stage == 'Pre-Training' else box_width/4

    xcent = x_center + offset
    # full range
    rect_minmax = Rectangle(
        (xcent - range_width/2, row['minv']),
        width=range_width,
        height=(row['maxv'] - row['minv']),
        facecolor=control_color,
        alpha=0, # 0.10
        edgecolor=None,
        zorder=0
    )
    ax.add_patch(rect_minmax)

    # IQR
    rect_iqr = Rectangle(
        (xcent - range_width/2, row['q1']),
        width=range_width,
        height=(row['q3'] - row['q1']),
        facecolor=control_color,
        alpha=0, # 0.18
        edgecolor=None,
        zorder=0.5
    )
    ax.add_patch(rect_iqr)

    # median
    ax.hlines(y=row['median'],
              xmin=xcent - range_width/2,
              xmax=xcent + range_width/2,
              linewidth=2.5,
              color=control_color,
              alpha=0, # 0.9
              zorder=1)

# -----------------------
# Test boxplot (test only)
# -----------------------
sns.boxplot(
    x='amb',
    y=y_col,
    data=df_test,
    hue='Stage',
    showfliers=False,
    width=box_width,
    linewidth=1.5,
    palette={'Post-Training': colors['Post-Training'], 'Pre-Training': colors['Pre-Training']},
    ax=ax,
    zorder=3
)


control_median_all = df_control['ABC_full'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,      # same blue as control shading
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)

# -----------------------
# Manual scatter for test points (deterministic dodge + jitter)
# so we can draw perfect pre/post lines
# -----------------------
amb_to_x = {amb: i for i, amb in enumerate(amb_order)}
hue_centers = {'Pre-Training': -box_width/4, 'Post-Training': box_width/4}

df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['amb'].map(amb_to_x).astype(float)
df_test['x_offset'] = df_test['Stage'].map(hue_centers).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

# scatter per stage so colours match
for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(sub['x_plot'], sub[y_col],
               color=colors[stage],
               edgecolors='black',
               linewidths=0.8,
               s=40,
               alpha=0.95,
               zorder=5,
               label=None)

# -----------------------
# Pair lines (connect Pre -> Post per (ID, amb))
# -----------------------
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    # require both stages
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-Training'].iloc[0]
        ax.plot([pre['x_plot'], post['x_plot']],
                [pre[y_col], post[y_col]],
                color='black', alpha=0.4, linewidth=1, zorder=4)

# -----------------------
# Aesthetic tweaks
# -----------------------
ax.set_ylabel("Area Under the Curve (AUC)", fontsize = 14)
ax.set_xlabel("")
ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order, fontsize = 14)
ax.set_title('Pre- and Post Training Area Under Contrast Sensitivity Curves', fontsize = 15)
# set y ticks intelligently
# ymax = int(np.nanmax(df_all[y_col]) * 1.05)
# ytick_step = int(max(1, round(ymax / 12)))
# ax.set_yticks(range(0, ymax + ytick_step, ytick_step))
ax.set_yticks(range(0, 3000, 500))
ax.set_ylim(0, 3000)
# legend: keep single legend for stage colours
handles = [Line2D([0], [0], color=colors['Pre-Training'], lw=5),
           Line2D([0], [0], color=colors['Post-Training'], lw=5), 
           Line2D([0], [0], color=control_color, alpha = 0.35, lw=5)]
ax.legend(handles=handles, labels=['Pre-Training', 'Post-Training', 'Control'],
          loc='lower left', handlelength=3)

# Example of adding a significance bar
x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
y, h, col = 2700, 50, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col, fontsize = 13)
h2 = 2500
ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x1+x1)*.5, h2+h+20, "n.s.", ha='center', va='bottom', color=col)

ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
ax.text((x2+x2)*.5, h2+h+20, "n.s.", ha='center', va='bottom', color=col)

plt.tight_layout()
plt.savefig('../Data/Plots/new_new/ABC_AUC_plot.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# -----------------------
# CONFIG / pick y column
# -----------------------
y_col = 'ABC_full' if 'ABC_full' in df_long_clean_BE.columns else ('ABC' if 'ABC' in df_long_clean_BE.columns else None)
if y_col is None:
    raise ValueError("Couldn't find 'ABC_full' or 'ABC' in df_long_clean_BE columns.")

box_width = 0.6
range_width = 0.36   # width (in x data units) for the control vertical boxes
jitter = 0.06
rng = np.random.default_rng(42)

# colours / styles
colors = {'Pre-Training': '#D55E00', 'Post-training': '#009E73'}
control_color = '#1E90FF'
linestyles = {'Both Eyes': '-'}

# -----------------------
# Prepare dataframes
# -----------------------
df_all = df_long_clean_BE.copy()

# ensure ID is string so grouping is stable
df_all['ID'] = df_all['ID'].astype(str)

# ensure Stage names are exactly as we expect
# if there are small differences (e.g., 'Post-training' vs 'Posttraining') this won't fix them,
# but it's easy to adapt if needed.
stage_order = ['Pre-Training', 'Post-training']

# amb order from categorical if available, else infer
if pd.api.types.is_categorical_dtype(df_all['amb']):
    amb_order = list(df_all['amb'].cat.categories)
else:
    amb_order = sorted(df_all['amb'].unique(), key=str)
    df_all['amb'] = pd.Categorical(df_all['amb'], categories=amb_order, ordered=True)

# split test vs control
df_test = df_all[df_all['group'] == 'test'].reset_index(drop=True)
df_control = df_all[df_all['group'] == 'control'].reset_index(drop=True)

# -----------------------
# Control stats (per amb × Stage × SF equivalent)
# We'll compute per-(amb,Stage) min, q1, median, q3, max of y_col
# -----------------------
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'], as_index=False)[y_col]
    .agg(q1=lambda x: x.quantile(0.25),
         median='median',
         q3=lambda x: x.quantile(0.75),
         minv='min',
         maxv='max',
         n='count')
)

# -----------------------
# Make figure
# -----------------------
sns.set_theme(style="white", palette="pastel")
fig, ax = plt.subplots(figsize=(4, 6))

# draw control vertical ranges behind boxes
for _, row in ctrl_stats.iterrows():
    amb = row['amb']
    stage = row['Stage']
    # x center = index of amb + small offset depending on stage
    x_center = amb_order.index(amb)
    # offsets for pre/post (match how boxplot/hue dodge will place them)
    offset = -box_width/4 if stage == 'Pre-Training' else box_width/4

    xcent = x_center + offset
    # full range
    rect_minmax = Rectangle(
        (xcent - range_width/2, row['minv']),
        width=range_width,
        height=(row['maxv'] - row['minv']),
        facecolor=control_color,
        alpha=0.10,
        edgecolor=None,
        zorder=0
    )
    ax.add_patch(rect_minmax)

    # IQR
    rect_iqr = Rectangle(
        (xcent - range_width/2, row['q1']),
        width=range_width,
        height=(row['q3'] - row['q1']),
        facecolor=control_color,
        alpha=0.18,
        edgecolor=None,
        zorder=0.5
    )
    ax.add_patch(rect_iqr)

    # median
    ax.hlines(y=row['median'],
              xmin=xcent - range_width/2,
              xmax=xcent + range_width/2,
              linewidth=2.5,
              color=control_color,
              alpha=0.9,
              zorder=1)

# -----------------------
# Test boxplot (test only)
# -----------------------
sns.boxplot(
    x='amb',
    y=y_col,
    data=df_test,
    hue='Stage',
    showfliers=False,
    width=box_width,
    linewidth=1.5,
    palette={'Post-training': colors['Post-training'], 'Pre-Training': colors['Pre-Training']},
    ax=ax,
    zorder=3
)


control_median_all = df_control['ABC_full'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,      # same blue as control shading
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)

# -----------------------
# Manual scatter for test points (deterministic dodge + jitter)
# so we can draw perfect pre/post lines
# -----------------------
amb_to_x = {amb: i for i, amb in enumerate(amb_order)}
hue_centers = {'Pre-Training': -box_width/4, 'Post-training': box_width/4}

df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['amb'].map(amb_to_x).astype(float)
df_test['x_offset'] = df_test['Stage'].map(hue_centers).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

# scatter per stage so colours match
for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(sub['x_plot'], sub[y_col],
               color=colors[stage],
               edgecolors='black',
               linewidths=0.8,
               s=40,
               alpha=0.95,
               zorder=5,
               label=None)

# -----------------------
# Pair lines (connect Pre -> Post per (ID, amb))
# -----------------------
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    # require both stages
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]
        ax.plot([pre['x_plot'], post['x_plot']],
                [pre[y_col], post[y_col]],
                color='black', alpha=0.4, linewidth=1, zorder=4)

# -----------------------
# Aesthetic tweaks
# -----------------------
ax.set_ylabel("Area Under the Curve (AUC)")
ax.set_xlabel("")
ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)

# set y ticks intelligently
# ymax = int(np.nanmax(df_all[y_col]) * 1.05)
# ytick_step = int(max(1, round(ymax / 12)))
# ax.set_yticks(range(0, ymax + ytick_step, ytick_step))
ax.set_yticks(range(0, 3000, 500))

# legend: keep single legend for stage colours
handles = [Line2D([0], [0], color=colors['Pre-Training'], lw=2.5),
           Line2D([0], [0], color=colors['Post-training'], lw=2.5)]
ax.legend(handles=handles, labels=['Pre-Training', 'Post-training'],
          loc='lower left', handlelength=3)

# # Example of adding a significance bar
# x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
# y, h, col = 2800, 50, 'black'
# ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# ax.text((x1+x2)*.5, y+h+0.2, "***", ha='center', va='bottom', color=col)
# h2 = 2600
# ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
# ax.text((x1+x1)*.5, h2+h+20, "n.s.", ha='center', va='bottom', color=col)

# ax.plot([x2-0.15, x2-0.15, x2+0.15, x2+0.15], [h2, h2+h, h2+h, h2], lw=1.5, c=col)
# ax.text((x2+x2)*.5, h2+h+20, "n.s.", ha='center', va='bottom', color=col)

plt.tight_layout()
# plt.savefig('../Data/Plots/Narrative_plots/ABC_AUC_plot.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
df_amb = df.copy()
df_amb = df_amb[(df_amb['amb'] == 'Lazy Eye')]

In [ ]:

offsets = {
    'P10': (0, -50),
    'P13': (50, 0)
}

fig, ax = plt.subplots(figsize=(4, 4))

for _, row in df_amb.iterrows():
    x = row['pretraining']
    y = row['posttraining']
    id_ = row['ID']

    if id_ in offsets:
        dx, dy = offsets[id_]

        # draw a small dot at the true data point only for shifted labels
        ax.plot(x, y, 'ko', markersize=3, zorder=3)

        ha = 'left' if dx > 0 else ('right' if dx < 0 else 'center')
        va = 'bottom' if dy > 0 else ('top' if dy < 0 else 'center')

        ax.annotate(
            str(id_),
            xy=(x, y),
            xytext=(x + dx, y + dy),
            fontsize=14,
            ha=ha, va=va,
            xycoords='data',
            textcoords='data',
            arrowprops=dict(
                arrowstyle='-',
                linewidth=1.0,
                color='black',
                shrinkA=6,   # shortens line so it doesn't go under the text
                shrinkB=0
            ),
            zorder=4
        )
    else:
        # regular labels: no dot, no connector
        ax.text(x, y, str(id_), fontsize=14, ha='center', va='center', zorder=4)

# reference line y = x
xmin = df_amb['pretraining'].min() - 200
xmax = df_amb['pretraining'].max() + 300
ax.plot([xmin, xmax], [xmin, xmax], '--', color='gray')

ax.set_xlim(50, 1500)
ax.set_ylim(50, 1500)
ax.set_xticks(range(0, 1500, 250))
ax.set_yticks(range(0, 1500, 250))
ax.set_xlabel("Pre-training AUC")
ax.set_ylabel("Post-training AUC")
ax.set_title("Lazy Eye AUC (pre- vs post- training)")
ax.grid(True)

plt.tight_layout()
plt.savefig('../Data/Plots/Narrative_plots/contrast_amb_AUC_pre_vs_post.png', dpi=600, bbox_inches='tight')

# plt.show()


## Interocular imbalance

The relative contrast needed in each eye for balanced (dichoptic) perception - i.e. how
much the amblyopic eye is being suppressed by the fellow eye, summarised as logBP.

In [ ]:
df = bv_diff['binoc'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Both': 'Both Eyes'})
df_long = df.melt(id_vars=['ID', 'Staircase', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logBP')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})

#for the control
df_con = bv_diff_con['binoc'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Both': 'Both Eyes'})
df_con_long = df_con.melt(id_vars=['ID', 'Staircase', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logBP')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)

In [ ]:

aov = pg.rm_anova(dv='logBP', within=['Stage','Staircase'], subject='ID', data=df_long_total[(df_long_total['group'] != 'control')])
aov.round(3)



In [ ]:
aov = pg.rm_anova(dv='logBP', within=['Stage','Staircase'], subject='ID', data=df_long_total[(df_long_total['group'] != 'test')])
aov.round(3)

In [ ]:
import pandas as pd
import pingouin as pg

# Make sure your dataframe is called df and columns match exactly:
# 'ID', 'Staircase', 'Stage', 'logBP'

# Convert factors to categorical
df_long['ID'] = df_long['ID'].astype('category')
df_long['Stage'] = df_long['Stage'].astype('category')
df_long['Staircase'] = df_long['Staircase'].astype('category')

# Run two-way repeated measures ANOVA
aov = pg.rm_anova(
    dv='logBP',
    within=['Stage', 'Staircase'],
    subject='ID',
    data=df_long,
    detailed=True
)

print(aov)


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Define the tick labels (assuming Staircase is 1 to 4)
sf_labels = ['0.5', '1', '2', '4']
df_long['Staircase'] = df_long['Staircase'].astype(int)  # ensure int for sorting

# Desired left->right order for the *hue* categories
stage_order = ['Pre-Training', 'Post-training']
hue_order = stage_order  # clearer name for use with hue

# Set theme
sns.set_theme(style="whitegrid", palette="pastel")

# Create figure and axis
fig, ax = plt.subplots(figsize=(7, 4))

# Boxplot: DON'T pass order=stage_order (that's for x categories). Use hue_order instead.
sns.boxplot(
    data=df_long,
    x="Staircase",
    y="logBP",
    hue="Stage",
    hue_order=hue_order,
    showfliers=False,
    width=0.6,
    linewidth=1.5,
    palette={'Pre-Training': '#D55E00', 'Post-training': '#009E73'},
    dodge=True,    # separate boxes per hue
    ax=ax
)

# Overlay stripplot (same hue_order so points line up with the boxes)
sns.stripplot(
    data=df_long,
    x="Staircase",
    y="logBP",
    hue="Stage",
    hue_order=hue_order,
    dodge=True,
    edgecolor='black',
    linewidth=1,
    palette={'Pre-Training': '#D55E00', 'Post-training': '#009E73'},
    ax=ax,
    legend=False,  # suppress extra legend from the stripplot
    jitter=0.15    # optional: gives nicer horizontal spread for points
)

# If seaborn left an automatic legend from boxplot, remove it because you add a manual one:
if ax.get_legend() is not None:
    ax.get_legend().remove()

# Y ticks / formatting
ax.set_yticks([0.5, 1.5, 3])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# X ticks: seaborn places integer Staircase categories at positions 0..(n-1)
ax.set_xticks(np.arange(len(sf_labels)))
ax.set_xticklabels(sf_labels)

# Axis labels
ax.set_xlabel("Spatial Frequency (cycles per dva)")
ax.set_ylabel("log of Balance Point (logBP)")

# Manual legend
legend_elements = [
    Patch(facecolor='#D55E00', edgecolor='black', label='Pre-Training'),
    Patch(facecolor='#009E73', edgecolor='black', label='Post-training'), 
    Line2D([0], [0], color='blue', lw=3, linestyle='--', label='Balance Value')
]
ax.legend(
    handles=legend_elements,
    title='Stage',
    loc='upper right',
    frameon=True,
    fontsize=10,
    title_fontsize=11
)

# Add horizontal line at y=0.5 behind all plot elements
ax.axhline(y=0.5, color='blue', linestyle='--', linewidth=3, zorder=1)
# plt.title('Significant Improvement in Interocular Balance in Amblyopes', fontsize=18)
plt.tight_layout()
# plt.show()
#plt.savefig('../Data/Plots/Narrative_plots/imbalance.png', dpi=600, bbox_inches='tight')


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------
# Config
# -----------------------
df = df_long_total.copy()
y_col = 'logBP'

stair_order = [1, 2, 3, 4]
stair_labels = ['0.5', '1', '2', '4']

stage_order = ['Pre-Training', 'Post-training']
colors = {'Pre-Training': '#D55E00', 'Post-training': '#009E73'}

jitter = 0.08
rng = np.random.default_rng(42)

# ensure types
df['Staircase'] = df['Staircase'].astype(int)
df['ID'] = df['ID'].astype(str)
df_control = df[df['group'] == 'control'].copy()
df = df[df['group'] == 'test'].copy()

# -----------------------
# Figure
# -----------------------
sns.set_theme(style="whitegrid", palette="pastel")
fig, ax = plt.subplots(figsize=(8, 5.5))

# -----------------------
# Scatter points (test only, alpha=0.3)
# -----------------------
stair_to_x = {s: i for i, s in enumerate(stair_order)}
hue_offsets = {'Pre-Training': -0.15, 'Post-training': 0.15}

df['x_base'] = df['Staircase'].map(stair_to_x).astype(float)
df['x_offset'] = df['Stage'].map(hue_offsets).astype(float)
df['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df)).astype(float)
df['x_plot'] = df['x_base'] + df['x_offset'] + df['x_jitter']

for stage in stage_order:
    sub = df[df['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub[y_col],
        color=colors[stage],
        edgecolors='black',
        linewidths=0.6,
        s=40,
        alpha=0.2,
        zorder=3
    )

# -----------------------
# Linear fits ± SD (test population)
# -----------------------
x_dense = np.linspace(0.9, 4.1, 300)

for stage in stage_order:
    sub = df[df['Stage'] == stage]
    if sub.shape[0] < 3:
        continue

    x = sub['Staircase'].values.astype(float)
    y = sub[y_col].values.astype(float)

    # linear fit
    coeffs = np.polyfit(x, y, 1)
    y_fit_dense = np.polyval(coeffs, x_dense)
    y_fit_obs = np.polyval(coeffs, x)

    # SD of residuals
    sd = (y - y_fit_obs).std(ddof=1)

    # plot fit
    ax.plot(
        x_dense - 1,      # map Staircase 1..4 → x positions 0..3
        y_fit_dense,
        color=colors[stage],
        linewidth=2.8,
        zorder=5
    )

    # SD band
    ax.fill_between(
        x_dense - 1,
        y_fit_dense - sd,
        y_fit_dense + sd,
        color=colors[stage],
        alpha=0,
        zorder=4
    )


# Linear fit of controls

x = df_control['Staircase'].values.astype(float)
y = df_control[y_col].values.astype(float)

 # linear fit
coeffs = np.polyfit(x, y, 1)
y_fit_dense = np.polyval(coeffs, x_dense)
y_fit_obs = np.polyval(coeffs, x)

# SD of residuals
sd = (y - y_fit_obs).std(ddof=1)

# plot fit
ax.plot(
    x_dense - 1,      # map Staircase 1..4 → x positions 0..3
    y_fit_dense,
    color='#1E90FF',
    linestyle='-',
    alpha = 1,
    linewidth=2.8,
    zorder=5
)

# -----------------------
# Horizontal balance reference line
# -----------------------
# ax.axhline(
#     y=0.5,
#     color='black',
#     linestyle='--',
#     linewidth=3,
#     alpha=0.5,
#     zorder=1
# )

# -----------------------
# Axes formatting
# -----------------------
ax.set_xticks(np.arange(len(stair_labels)))
ax.set_xticklabels(stair_labels)
ax.set_xlabel("Spatial Frequency (cycles per dva)", fontsize = 14)
ax.set_ylabel("log of Balance Point (logBP)", fontsize = 14)
ax.set_title("Balance Point", fontsize = 15)
ymin = 0#np.floor(df[y_col].min() - 0.5)
ymax = 3#np.ceil(df[y_col].max() + 0.5)
ax.set_ylim(ymin, ymax)
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# -----------------------
# Legend
# -----------------------
legend_handles = [
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor=colors['Pre-Training'], markeredgecolor='black',
           markersize=8, alpha = 0.2, label='Pre-Training (points)'),
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor=colors['Post-training'], markeredgecolor='black',
           markersize=8, alpha = 0.2, label='Post-training (points)'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.8, label='Pre-Training (linear fit)'),
    Line2D([0], [0], color=colors['Post-training'], lw=2.8, label='Post-Training (linear fit)'),
    Line2D([0], [0], color='#1E90FF', lw=2.8, label='Controls (linear fit)'),
    #Line2D([0], [0], color='black', alpha = 0.5, lw=3, linestyle='--', label='Balance value (0.5)')
]

ax.legend(handles=legend_handles, loc='upper right', frameon=True, fontsize=13)

# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
# plt.savefig(
#     '../Data/Plots/new_new/logBP_linear_fits.png',
#     dpi=600,
#     bbox_inches='tight'
# )
plt.show()


In [ ]:
bv_auc = pd.read_excel(path_data_ref+'/bv_4_results_ABC_diff.xlsx', sheet_name=None) 
bv_auc_con = pd.read_excel(path_data_ref_con+'/bv_4_results_ABC_diff.xlsx', sheet_name=None)


In [ ]:
df = bv_auc['abc_df_bi'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Both': 'Both Eyes'})
df_long = df.melt(id_vars=['ID', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='ABC_full')
#df_long['amb'] = df_long['amb'].map({'Amb': 'Lazy Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})
print(df_long)

# for the controls
df_con = bv_auc_con['abc_df_bi'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Both': 'Both Eyes'})
df_con_long = df_con.melt(id_vars=['ID', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='ABC_full')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)


In [ ]:
df_avg_con = (
    df_con_long
    .groupby(['ID', 'group'], as_index=False)
    .agg(mean_value=('ABC_full', 'mean'))
)

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Pre-Training']['ABC_full'].values,
    y=df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['ABC_full'].values
)

print(ttest.round(3))
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Pre-Training']['ABC_full'].mean())
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Pre-Training']['ABC_full'].std())
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['ABC_full'].mean())
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['ABC_full'].std())


In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_total[(df_long_total['group'] != 'test')][df_long_total[(df_long_total['group'] != 'test')]['Stage'] == 'Pre-Training']['ABC_full'].values,
    y=df_long_total[(df_long_total['group'] != 'test')][df_long_total[(df_long_total['group'] != 'test')]['Stage'] == 'Post-Training']['ABC_full'].values
)

print(ttest.round(3))

In [ ]:
# Run paired t-test between control and test
ttest = pg.ttest(
    paired=False,
    x=df_con_long['ABC_full'].values,
    y=df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['ABC_full'].values
)

#     y=df_long_total[(df_long_total['group'] != 'test')][df_long_total[(df_long_total['group'] != 'test')]['Stage'] == 'Pre-Training']['ABC_full'].values


print(ttest.round(3))
print(df_long_total[(df_long_total['group'] != 'test')][df_long_total[(df_long_total['group'] != 'test')]['Stage'] == 'Post-Training']['ABC_full'].mean())
print(df_long_total[(df_long_total['group'] != 'test')][df_long_total[(df_long_total['group'] != 'test')]['Stage'] == 'Post-Training']['ABC_full'].std())


### Area Under Curve

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# -------------------------
# Config / pick y column
# -------------------------
y_col = 'ABC_full'   # change if your column has a different name
if y_col not in df_long.columns:
    raise ValueError(f"Column '{y_col}' not found in df_long")

# plotting params
box_width = 0.4
range_width = 0.5   # horizontal width (in x data units) of control vertical ranges
jitter = 0.3
rng = np.random.default_rng(42)

colors = {'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
control_color = '#1E90FF'
stage_order = ['Pre-Training', 'Post-Training']  # ensures left->right order

# -------------------------
# Prepare dataframes
# -------------------------
df = df_long_total.copy()
df['ID'] = df['ID'].astype(str)
df['Stage'] = df['Stage'].astype(str)

df_test = df[df['group'] == 'test'].reset_index(drop=True)
df_control = df[df['group'] == 'control'].reset_index(drop=True)

# -------------------------
# Control stats per Stage (min, q1, median, q3, max)
# -------------------------
ctrl_stats = (
    df_control
    .groupby('Stage', as_index=False)[y_col]
    .agg(
        mean='mean',
        sd='std',
        n='count'
    )
)

# enforce stage order
ctrl_stats['Stage'] = pd.Categorical(ctrl_stats['Stage'],
                                     categories=stage_order,
                                     ordered=True)
ctrl_stats = ctrl_stats.sort_values('Stage')

# -------------------------
# Figure
# -------------------------
sns.set_theme(style="white", palette="pastel")
fig, ax = plt.subplots(figsize=(4, 7))

# Draw control shaded vertical ranges behind boxes
# map Stage -> x center (0, 1, ...)
stage_to_x = {stage: i for i, stage in enumerate(stage_order)}

# for _, row in ctrl_stats.iterrows():
#     stage = row['Stage']
#     if pd.isna(stage):
#         continue

#     x_center = stage_to_x[stage]

#     mean = row['mean']
#     sd = row['sd']

#     # mean ± SD shaded band
#     rect_sd = Rectangle(
#         (x_center - range_width/2, mean - sd),
#         width=range_width,
#         height=2 * sd,
#         facecolor=control_color,
#         alpha=0.18,
#         edgecolor=None,
#         zorder=0
#     )
#     ax.add_patch(rect_sd)

#     # mean line
#     ax.hlines(
#         y=mean,
#         xmin=x_center - range_width/2,
#         xmax=x_center + range_width/2,
#         linewidth=3,
#         color=control_color,
#         alpha=0.95,
#         zorder=1
#     )


# -------------------------
# Test boxplot (test only)
# -------------------------
# ensure categorical Stage order so seaborn places them left->right
df_test['Stage_cat'] = pd.Categorical(df_test['Stage'], categories=stage_order, ordered=True)

sns.boxplot(
    data=df_test,
    x='Stage_cat',
    y=y_col,
    showfliers=False,
    width=box_width,
    linewidth=1.5,
    palette={stage_order[0]: colors[stage_order[0]], stage_order[1]: colors[stage_order[1]]},
    ax=ax,
    zorder=3
)

# -------------------------
# Manual scatter for test points (deterministic dodge + jitter)
# -------------------------
hue_offsets = {'Pre-Training': -box_width/4, 'Post-Training': box_width/4}
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['Stage'].map(stage_to_x).astype(float)
df_test['x_offset'] = df_test['Stage'].map(hue_offsets).astype(float)


# df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_jitter'] = 0.0

for stage in stage_order:
    mask = df_test['Stage'] == stage
    n = mask.sum()
    
    if n > 1:
        spread = np.linspace(-jitter, jitter, n)
        df_test.loc[mask, 'x_jitter'] = spread

df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

# scatter points (on top of boxes)
# for stage in stage_order:
#     sub = df_test[df_test['Stage'] == stage]
#     ax.scatter(
#         sub['x_plot'],
#         sub[y_col],
#         color=colors[stage],
#         edgecolors='black',
#         linewidths=0.8,
#         s=40,
#         alpha=0.95,
#         zorder=6,
#         label=None
#     )

# plot ID numbers instead of dots
for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    for _, row in sub.iterrows():
        ax.text(
            row['x_plot'],
            row[y_col],
            row['ID'],
            fontsize=13,
            ha='center',
            va='center',
            zorder=7
        )

# -------------------------
# Pair lines connecting Pre->Post for test subjects (if both present)
# -------------------------
for pid, g in df_test.groupby('ID'):
    # require both stages present for this ID
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-Training'].iloc[0]
        ax.plot([pre['x_plot'], post['x_plot']],
                [pre[y_col], post[y_col]],
                color='black', alpha=0.4, linewidth=1, zorder=5)


control_median_all = df_control['ABC_full'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,      # same blue as control shading
    linestyle='--',
    linewidth=3,
    alpha=0.5,
    zorder=1
)

# -------------------------
# Aesthetic tweaks
# -------------------------
ax.set_ylabel("Area Under the Linear fit (AUL)", fontsize = 14)
ax.set_xlabel("")
ax.set_xticks(list(stage_to_x.values()))
ax.set_xticklabels(stage_order, fontsize = 14)
ax.set_title('Area Under Linear Fit', fontsize = 15)
# y ticks: choose sensible spacing
ymin = float(np.nanmin(df[y_col])) if not df[y_col].isna().all() else 0.0
ymax = 10 # float(np.nanmax(df[y_col])) if not df[y_col].isna().all() else 1.0
yrange = ymax - ymin
pad = max(0.1 * yrange, 0.5)
ax.set_ylim(ymin - pad*0.15, ymax + pad*0.2)

# create yticks nicely
ytick_step = max(1, int(round((ymax - ymin) / 6))) if (ymax - ymin) >= 2 else None
if ytick_step:
    ax.set_yticks(np.arange(int(np.floor(ymin)), int(np.ceil(ymax + 1)), ytick_step))
else:
    ax.yaxis.set_major_locator(plt.MaxNLocator(6))
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# -------------------------
# Combined legend: Stage colours + control envelope
# -------------------------
legend_handles = [
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training'),
    # Line2D([], [], linestyle='none', label=''),
    Line2D([0], [0], color=control_color, lw=2.5, linestyle='--', label='  Control', alpha=0.5)
]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, fontsize=12, handlelength=3)

# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)
# -------------------------
# Optional significance bar example (edit coordinates/labels as desired)
# -------------------------
x1, x2 = 0, 1
y_sig = 8.3 + pad*0.06
h = pad*0.03
ax.plot([x1, x1, x2, x2], [y_sig, y_sig+h, y_sig+h, y_sig], lw=1.5, c='black')
ax.text((x1+x2)*0.5, y_sig+h*1.2, "*", ha='center', va='bottom', color='black')

plt.tight_layout()
plt.savefig('../Data/Plots/new_new/imbalance_AUC.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
offsets = {
    'P04': (-0.3, 0.3),
    'P13': (0.5, -0.1), 
    'P20': (-0.5, -0.1),
    'P05': (0, -0.5),
    'P16': (0.5, -0.1),
    'P14': (0, 0.5),
    'P18': (-0.2, -0.5)
}

fig, ax = plt.subplots(figsize=(4, 4))

for _, row in df.iterrows():
    x = row['pretraining']
    y = row['posttraining']
    id_ = row['ID']

    if id_ in offsets:
        dx, dy = offsets[id_]

        # draw a small dot at the true data point only for shifted labels
        ax.plot(x, y, 'ko', markersize=3, zorder=3)

        ha = 'left' if dx > 0 else ('right' if dx < 0 else 'center')
        va = 'bottom' if dy > 0 else ('top' if dy < 0 else 'center')

        ax.annotate(
            str(id_),
            xy=(x, y),
            xytext=(x + dx, y + dy),
            fontsize=14,
            ha=ha, va=va,
            xycoords='data',
            textcoords='data',
            arrowprops=dict(
                arrowstyle='-',
                linewidth=1.0,
                color='black',
                shrinkA=6,   # shortens line so it doesn't go under the text
                shrinkB=0
            ),
            zorder=4
        )
    else:
        # regular labels: no dot, no connector
        ax.text(x, y, str(id_), fontsize=14, ha='center', va='center', zorder=4)

# reference line y = x
xmin = df['pretraining'].min() - 2
xmax = df['pretraining'].max() + 3
ax.plot([xmin, xmax], [xmin, xmax], '--', color='gray')

ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
ax.set_xticks(range(0, 6, 1))
ax.set_yticks(range(0, 6, 1))
ax.set_xlabel("Pre-training AUC")
ax.set_ylabel("Post-training AUC")
ax.set_title("Lazy Eye AUC (pre- vs post- training)")
ax.grid(True)

# plt.tight_layout()
plt.savefig('../Data/Plots/Narrative_plots/imbalance_amb_AUC_pre_vs_post.png', dpi=600, bbox_inches='tight')

# plt.show()


DO NOT FORGET TO MENTION THE JITTER FOR THIS ONE!

## Dichoptic Noise
How much noise in the fellow eye it takes to disrupt the amblyopic eye's perception,
another measure of interocular suppression.

In [ ]:
df = bv_diff['dichop'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Both': 'Both Eyes'})
df_long = df.melt(id_vars=['ID', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='Contrast')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

#for the control
df_con = bv_diff_con['dichop'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Both': 'Both Eyes'})
df_con_long = df_con.melt(id_vars=['ID', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='Contrast')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Pre-Training']['Contrast'].values,
    y=df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['Contrast'].values
)

print(ttest.round(3))
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Pre-Training']['Contrast'].mean())
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Pre-Training']['Contrast'].std())
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['Contrast'].mean())
print(df_long_total[(df_long_total['group'] != 'control')][df_long_total[(df_long_total['group'] != 'control')]['Stage'] == 'Post-Training']['Contrast'].std())


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ---------- Prepare df_long_total and unify value column ----------
df = df_long_total.copy()
df['Contrast'] = df['Contrast']*100
# possible value columns used earlier: 'Value', 'Contrast', 'Frequency', 'ABC_full', 'logBP'
value_candidates = ['Value', 'Contrast', 'Frequency', 'ABC_full', 'logBP']
value_col = None
for c in value_candidates:
    if c in df.columns:
        value_col = c
        break
if value_col is None:
    # fallback: try to detect numeric column aside from ID, group, Stage, amb
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        value_col = numeric_cols[0]
    else:
        raise ValueError("Couldn't find a value column in df_long_total")

# unify to 'Value'
if value_col != 'Value':
    df = df.rename(columns={value_col: 'Value'})

# # Normalize Stage and group strings
# df['Stage'] = df['Stage'].astype(str).str.replace('-', ' ').str.strip()
# df['Stage'] = df['Stage'].replace({
#     'pretraining': 'Pre-Training',
#     'pre_training': 'Pre-Training',
#     'Pretraining': 'Pre-Training',
#     'posttraining': 'Post-training',
#     'post_training': 'Post-training',
#     'Posttraining': 'Post-training'
# })
df['group'] = df['group'].astype(str).str.lower()  # 'test' / 'control' expected

# keep only the two stages we care about
stage_order = ['Pre-Training', 'Post-Training']
df = df[df['Stage'].isin(stage_order)].copy()

# ensure ID is string
df['ID'] = df['ID'].astype(str)

# split test vs control
df_test = df[df['group'] == 'test'].reset_index(drop=True)
df_control = df[df['group'] == 'control'].reset_index(drop=True)

# ---------- Compute control mean ± SD per Stage ----------
ctrl_stats = (
    df_control
    .groupby('Stage', as_index=False)['Value']
    .agg(mean='mean', sd='std', n='count')
)
# enforce stage ordering
ctrl_stats['Stage'] = pd.Categorical(ctrl_stats['Stage'], categories=stage_order, ordered=True)
ctrl_stats = ctrl_stats.sort_values('Stage')

# ---------- Plotting params ----------
sns.set_theme(style="white")
colors = {'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
control_color = '#1E90FF'
box_width = 0.35
range_width = 0.5
jitter = 0.06
rng = np.random.default_rng(42)

# ---------- Make figure ----------
fig, ax = plt.subplots(figsize=(7, 6))

# # Draw control mean ± SD rectangles behind everything
# stage_to_x = {stage: i for i, stage in enumerate(stage_order)}
# for _, row in ctrl_stats.iterrows():
#     stage = row['Stage']
#     if pd.isna(stage):
#         continue
#     x_center = stage_to_x[stage]
#     mean = row['mean']
#     sd = 0.0 if np.isnan(row['sd']) else float(row['sd'])

#     # mean ± SD rectangle
#     rect_sd = Rectangle(
#         (x_center - range_width/2, mean - sd),
#         width=range_width,
#         height=2 * sd,
#         facecolor=control_color,
#         alpha=0.18,
#         edgecolor=None,
#         zorder=0
#     )
#     ax.add_patch(rect_sd)

#     # mean horizontal line
#     ax.hlines(
#         y=mean,
#         xmin=x_center - range_width/2,
#         xmax=x_center + range_width/2,
#         linewidth=3,
#         color=control_color,
#         alpha=0.95,
#         zorder=1
#     )

# ---------- Control overall median horizontal line ----------
control_median_all = df_control['Value'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)

# ---------- Test-only boxplot ----------
# create ordered categorical for Stage so seaborn places them left->right
df_test['Stage_cat'] = pd.Categorical(df_test['Stage'], categories=stage_order, ordered=True)
sns.boxplot(
    data=df_test,
    x='Stage_cat',
    y='Value',
    showfliers=False,
    width=box_width,
    linewidth=1.5,
    palette={stage_order[0]: colors[stage_order[0]], stage_order[1]: colors[stage_order[1]]},
    ax=ax,
    zorder=3
)

# ---------- Manual scatter for test points (deterministic dodge + jitter) ----------
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['Stage'].map(stage_to_x).astype(float)
hue_offsets = {'Pre-Training': -box_width/4, 'Post-Training': box_width/4}
df_test['x_offset'] = df_test['Stage'].map(hue_offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['Value'],
        color=colors[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=6,
        label=None
    )

# ---------- Paired lines connecting Pre -> Post for test subjects ----------
for pid, g in df_test.groupby('ID'):
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-Training'].iloc[0]
        ax.plot([pre['x_plot'], post['x_plot']],
                [pre['Value'], post['Value']],
                color='black', alpha=0.4, linewidth=1, zorder=5)

# ---------- Aesthetics ----------
ax.set_xticks(list(stage_to_x.values()))
ax.set_xticklabels(stage_order)
ax.set_xlabel("")
ax.set_ylabel("Noise Contrast (%)")  # change label if you'd prefer the original name

# y-limits and ticks: add some padding
ymin = float(df['Value'].min())
ymax = float(df['Value'].max())
yrange = ymax - ymin if (ymax - ymin) != 0 else 1.0
pad = max(0.1 * yrange, 0.1)
ax.set_ylim(0, 100) # ymin - pad*0.15, ymax + pad*0.2
ax.yaxis.set_major_locator(plt.MaxNLocator(7))
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# ---------- Legend: Stage colours + control entry ----------
legend_handles = [
    # Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-Training'], lw=2.5, label='  Post-Training'),
    Line2D([0], [0], color=control_color, lw=2.5, alpha = 0.35, label='Control Median', linestyle='--')
]
ax.legend(handles=legend_handles, loc='upper center', frameon=True, fontsize=13, handlelength=3)

# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

ax.set_ylabel("Noise Contrast (%)", fontsize = 14)
ax.set_title('Pre- and Post-Training Dichoptic Noise Test', fontsize = 15)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.xticks([0, 1], ['Pre-Training', 'Post-Training'], fontsize = 14)



plt.tight_layout()
plt.savefig('../Data/Plots/new_new/noise_contrast_dichoptic.png', dpi=600, bbox_inches='tight')
plt.show()


## Asteroid
Stereoacuity (depth-perception threshold), measured in arcseconds of disparity.

### First the trials

In [ ]:
bv_trials = pd.read_excel(path_data_ref+'/bv_1_results_trials.xlsx', sheet_name=None) 
bv_trials_con = pd.read_excel(path_data_ref_con+'/bv_1_results_trials.xlsx', sheet_name=None) 


In [ ]:
df = bv_trials['df_as'].copy()
print(df.columns)

In [ ]:
df = bv_trials['df_as'].copy()
df['group'] = 'test'
indexID = df[(df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index
df.drop(indexID , inplace=True)
df_long = df.copy()
df_long['stage'] = df_long['stage'].map({'familiarisation':'Familiarisation', 'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

#for the control
df_con = bv_trials_con['df_as'].copy()
df_con['group'] = 'control'
df_con_long = df_con.copy()
df_con_long['stage'] = df_con_long['stage'].map({'familiarisation':'Familiarisation', 'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle


# -------------------------
# Prepare data
# -------------------------
# use df_temp as provided by you; filter to test group only
df = df_long_total.copy()
df_con = df[df['group'] == 'control'].reset_index(drop=True)
df = df[df['group'] == 'test'].reset_index(drop=True)
# df = df[df['stage'] == 'Pre-Training'].reset_index(drop=True)
df = df[
    ~(
        (df['stage'] == 'Familiarisation')
     )
]

df_con = df_con[
    (
        (df_con['stage'] == 'Post-Training')
    )
]

# IDs you want to exclude from the *post-training* box stats
post_ids = ['P08', 'P10', 'P17', 'P19', 'P21']
pre_ids = ['P08', 'P10']

# --- 1) Draw the boxplots from a filtered dataframe (stats ignore the 5 post-training points) ---
df_pre = df[
    ~(
        (df['stage'] == 'Post-Training') |
        (df['ID'].isin(post_ids))
     )
]

df_post = df[
    ~(
        (df['stage'] == 'Pre-Training') |
        (df['ID'].isin(post_ids))
    )
]

df_other = df[
    (
        (df['stage'] == 'Post-Training') &
        (df['ID'].isin(post_ids))
    )
]

# make sure amb values match expected strings; list unique to check if needed:
# print(df['amb'].unique())

# -------------------------
# Aggregate (means + sem + sd)
# -------------------------
agg_pre = (
    df_pre
    .groupby(['group', 'trials2'], as_index=False)
    .agg(
        mean_cs=('disparitydisparity_arcsec', 'mean'),
        sem_cs=('disparitydisparity_arcsec', 'sem'),
        std_cs=('disparitydisparity_arcsec', 'std')
    )
)
agg_pre['s'] = 'Pre-Training'

agg_post = (
    df_post
    .groupby(['group', 'trials2'], as_index=False)
    .agg(
        mean_cs=('disparitydisparity_arcsec', 'mean'),
        sem_cs=('disparitydisparity_arcsec', 'sem'),
        std_cs=('disparitydisparity_arcsec', 'std')
    )
)
agg_post['s'] = 'Post-training'

agg_other = (
    df_other
    .groupby(['group', 'trials2'], as_index=False)
    .agg(
        mean_cs=('disparitydisparity_arcsec', 'mean'),
        sem_cs=('disparitydisparity_arcsec', 'sem'),
        std_cs=('disparitydisparity_arcsec', 'std')
    )
)
agg_other['s'] = 'Others'


agg_con = (
    df_con
    .groupby(['group', 'trials2'], as_index=False)
    .agg(
        mean_cs=('disparitydisparity_arcsec', 'median'),
        sem_cs=('disparitydisparity_arcsec', 'sem'),
        std_cs=('disparitydisparity_arcsec', 'std')
    )
)

# merge by stacking rows (use concat, not merge)
agg = pd.concat([agg_pre, agg_post], ignore_index=True)
agg = pd.concat([agg, agg_other], ignore_index=True)

# -------------------------
# Plotting: log-parabola fits for each amb × stage with 95% CI
# -------------------------
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(7, 6))

amb_order = ['test', 'control']
stage_order = ['Others', 'Pre-Training', 'Post-training']
s_order = ['Pre-Training', 'Post-training', 'Others']
colors = {'Others':'#740EDA', 'Pre-Training': '#D55E00', 'Post-training': '#009E73'}

eps = 1e-8  # small positive number to avoid log(0)

# helper to fit log-parabola and return dense fit + CI
def fit_log_parabola(x_raw, y_raw, n_points=200, z=1.96):
    """
    Fit ln(y) = a2*(ln x)^2 + a1*(ln x) + a0 by np.polyfit (deg=2) with cov.
    Returns x_fit, y_fit, y_lower, y_upper (all on original y scale).
    If fit cannot be performed, raises Exception.
    """
    # ensure numpy arrays and positive values
    x_raw = np.asarray(x_raw).astype(float)
    y_raw = np.asarray(y_raw).astype(float)

    # require at least 3 distinct x points to fit a quadratic reliably
    if x_raw.size < 3 or len(np.unique(x_raw)) < 3:
        raise ValueError("Not enough distinct x points to fit quadratic.")

    # clip to tiny positive values
    x_safe = np.where(x_raw <= 0, eps, x_raw)
    y_safe = np.where(y_raw <= 0, eps, y_raw)

    lnx = np.log(x_safe)
    lny = np.log(y_safe)

    # fit quadratic in lnx
    coeffs, cov = np.polyfit(lnx, lny, deg=2, cov=True)  # coeffs: [a2, a1, a0]

    # dense x grid for smooth plotting (within range of data)
    x_fit = np.linspace(x_safe.min(), x_safe.max(), n_points)
    lnx_fit = np.log(np.where(x_fit <= 0, eps, x_fit))
    lny_fit = np.polyval(coeffs, lnx_fit)
    y_fit = np.exp(lny_fit)

    # gradient of polynomial w.r.t params [a2, a1, a0] at each lnx_fit
    grad = np.vstack([lnx_fit**2, lnx_fit, np.ones_like(lnx_fit)]).T  # shape (N,3)

    # variance of predicted ln(y): var = grad @ cov @ grad.T (diagonal elements)
    pred_var = np.einsum('ij,jk,ik->i', grad, cov, grad)
    # numerical issues could produce tiny negative variance; clip
    pred_var = np.clip(pred_var, a_min=0.0, a_max=None)
    pred_se = np.sqrt(pred_var)

    # 95% CI in log-space, then exponentiate
    lny_upper = lny_fit + z * pred_se
    lny_lower = lny_fit - z * pred_se
    y_upper = np.exp(lny_upper)
    y_lower = np.exp(lny_lower)

    return x_fit, y_fit, y_lower, y_upper

# Plot each group × stage: fit + 95% CI
for st in s_order:
    s_agg = agg[agg['s'] == st]
    for amb in amb_order:
        sub_agg = s_agg[s_agg['group'] == amb].sort_values('trials2')
        if sub_agg.empty:
            continue

        x = sub_agg['trials2'].values.astype(float)
        mean = sub_agg['mean_cs'].values.astype(float)

        # plot raw mean points faintly so user sees the observations
        ax.plot(x, mean, 'o', color=colors[st], alpha=0.35, markersize=6, zorder=5)

        # attempt to fit log-parabola to these mean points and plot fit + CI
        try:
            x_fit, y_fit, y_lower, y_upper = fit_log_parabola(x, mean, n_points=300, z=1.96)
            ax.plot(x_fit, y_fit,
                    linestyle='-',
                    linewidth=2.2,
                    color=colors[st],
                    alpha=0.95,
                    zorder=6,
                    label=f"{amb} — {st}")

            # shade 95% CI band (on original y scale)
            ax.fill_between(x_fit, y_lower, y_upper,
                            alpha=0.18,
                            facecolor=colors[st],
                            linewidth=0,
                            zorder=4)
        except Exception as e:
            # fallback: if fitting fails, plot a simple connecting line of the means
            ax.plot(x, mean,
                    linestyle='--',
                    linewidth=1.5,
                    color=colors[st],
                    alpha=0.75,
                    zorder=4,
                    label=f"{amb} — {st} (no fit)")
            # (optionally) you could also plot mean ± SD here if desired:
            # sd = sub_agg['std_cs'].values.astype(float)
            # ax.fill_between(x, np.clip(mean-sd, eps, None), mean+sd, alpha=0.12, facecolor=colors[st])
            # but per request we prefer fit+95%CI, so only fallback line is shown.

# -------------------------
# Also plot the control fit (you already had this) — keep as reference
# -------------------------
try:
    x_raw = agg_con['trials2'].values.astype(float)
    y_raw = agg_con['mean_cs'].values.astype(float)
    x_safe = np.where(x_raw <= 0, eps, x_raw)
    lnx = np.log(x_safe)
    lny = np.log(np.where(y_raw <= 0, eps, y_raw))
    coeffs, cov = np.polyfit(lnx, lny, deg=2, cov=True)
    x_fit_ctrl = np.linspace(x_safe.min(), x_safe.max(), 300)
    lnx_fit_ctrl = np.log(np.where(x_fit_ctrl <= 0, eps, x_fit_ctrl))
    lny_fit_ctrl = np.polyval(coeffs, lnx_fit_ctrl)
    y_fit_ctrl = np.exp(lny_fit_ctrl)
    grad = np.vstack([lnx_fit_ctrl**2, lnx_fit_ctrl, np.ones_like(lnx_fit_ctrl)]).T
    pred_var = np.einsum('ij,jk,ik->i', grad, cov, grad)
    pred_var = np.clip(pred_var, 0.0, None)
    pred_se = np.sqrt(pred_var)
    lny_upper = lny_fit_ctrl + 1.96 * pred_se
    lny_lower = lny_fit_ctrl - 1.96 * pred_se
    y_upper = np.exp(lny_upper)
    y_lower = np.exp(lny_lower)

    ax.plot(x_fit_ctrl, y_fit_ctrl, color='#1E90FF', lw=2.5, zorder=7, label='Controls (log-parabola fit)')
    ax.fill_between(x_fit_ctrl, y_lower, y_upper, color='#1E90FF', alpha=0.12, zorder=3)
except Exception:
    # ignore control fit failure
    pass

# -------------------------
# Formatting axes (log-scale Y)
# -------------------------
ax.set_yscale('log')
ax.set_ylim(10, 1100)
ax.set_yticks([10, 100, 1000])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax.set_xlabel('Trials', fontsize=14)
ax.set_ylabel('Disparity (arcseconds)', fontsize=14)

# Build a concise legend with one handle per stage + a controls handle
legend_handles = [
    # Line2D([], [], linestyle='none', label='Stage'),
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor='grey', markeredgecolor='black',
           markersize=8, alpha = 0.2, label='Mean Values'),
    Line2D([0], [0], color=colors['Pre-Training'], lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color=colors['Post-training'], lw=2.5, label='  Post-Training'),
    Line2D([0], [0], color=colors['Others'], lw=2.5, label='  Post-Training - Gained Stereopsis'),
    Line2D([0], [0], color='#1E90FF', lw=2.5, label='  Controls Mean')
]

ax.legend(handles=legend_handles, loc='lower left', frameon=True, fontsize=11.5, handlelength=3)

ax.tick_params(axis='both', width=2, length=6)
for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.set_title('Asteroid - Staircase Performance', fontsize=15)

plt.tight_layout()
plt.savefig('../Data/Plots/new_new/asteroid_staircase_fitCI.png',
             dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
agg_con['mean_cs'][60]

In [ ]:

df = bv_diff['asteroid'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') | (df['ID'] == 'P02') | (df['ID'] == 'P03')].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df_long = df.melt(id_vars=['ID', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='log arcseconds')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})
df_long['arcseconds'] = 10 ** df_long['log arcseconds']

#for the control
df_con = bv_diff_con['asteroid'].copy()
df_con['group'] = 'control'
df_con_long = df_con.melt(id_vars=['ID', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='log arcseconds')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})
df_con_long['arcseconds'] = 10 ** df_con_long['log arcseconds']

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)

In [ ]:
df = bv_diff['asteroid'].copy()[['ID','familiarisation', 'pretraining', 'posttraining', 'Diff']]
indexID = df[ (df['ID'] == 'P01')| (df['ID'] == 'P02') | (df['ID'] == 'P03')].index # | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df_2 = df.drop('Diff', axis=1)

In [ ]:
cols_to_replace = ['familiarisation', 'pretraining', 'posttraining']
#df = df.dropna()
df_2[cols_to_replace] = df_2[cols_to_replace].replace([3.0], np.nan)
#df_2 = df_2[df_2['pretraining'].notna()]
print(df_2)



In [ ]:
df_spec = df_2.dropna(subset = ['pretraining'], inplace=False)

In [ ]:
print(df_spec['pretraining'].mean())
print(df_spec['pretraining'].std())

In [ ]:
df_2bis = df.copy()
indexID = df_2bis[ (df['ID'] == 'P10') | (df_2bis['ID'] == 'P19') | (df_2bis['ID'] == 'P21') | (df_2bis['ID'] == 'P17') | (df_2bis['ID'] == 'P08')].index # | (df['ID'] == 'P03')
df2_bis = df_2bis.filter(items=indexID, axis=0)
# df_2 = df.drop('Diff', axis=1)


In [ ]:
print(df2_bis['posttraining'].mean())
print(df2_bis['posttraining'].std())

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_spec['pretraining'].values,
    y=df_spec['posttraining'].values
)
print(ttest.round(3))

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_spec['pretraining'].values,
    y=df_spec['posttraining'].values
)
print(ttest.round(3))

In [ ]:
df_3 = df_2.copy()
cols_to_replace = ['familiarisation', 'pretraining', 'posttraining']

df_3[cols_to_replace] = df_3[cols_to_replace].fillna(3.0)
print(df_3.head())

In [ ]:
# Run paired t-test
wilcox = pg.wilcoxon(
    x=df_3['pretraining'].values,
    y=df_3['posttraining'].values
)

print(wilcox.round(3))
print(df_3['posttraining'].mean())
print(df_3['posttraining'].std())
print(df_3['pretraining'].mean())
print(df_3['pretraining'].std())

In [ ]:
# Run paired t-test
wilcox = pg.wilcoxon(
    x=df_3['familiarisation'].values,
    y=df_3['pretraining'].values
)

print(wilcox.round(3))

wilcox = pg.wilcoxon(
    x=df_3['familiarisation'].values,
    y=df_3['posttraining'].values
)

print(wilcox.round(3))

In [ ]:
import numpy as np
import pandas as pd

THRESH = 3.0   # NaN means value > threshold

def label_change(pre, post, thresh=THRESH):
    """
    Returns:
      +1 -> improvement
      -1 -> decline
       0 -> tie
      None -> unknown
    """
    pre_c = np.isnan(pre)     # censored: pre > threshold
    post_c = np.isnan(post)   # censored: post > threshold

    # Case 1: both observed numeric values
    if not pre_c and not post_c:
        if post < pre: return +1
        if post > pre: return -1
        return 0

    # Case 2: pre is censored (>thresh) but post is observed
    if pre_c and not post_c:
        # post ≤ thresh means DEFINITE improvement
        if post <= thresh:
            return +1
        else:
            return None

    # Case 3: post is censored (>thresh) but pre is observed
    if not pre_c and post_c:
        # pre ≤ thresh → DEFINITE decline
        if pre <= thresh:
            return -1
        else:
            return None

    # Case 4: both censored → cannot tell direction
    return 0

# Apply to your DataFrame
df_2['change_sign'] = df_2.apply(lambda r: label_change(r['pretraining'], r['posttraining']), axis=1)

print(df_2)


In [ ]:
improve = (df_2['change_sign'] == 1).sum()
decline = (df_2['change_sign'] == -1).sum()
ties = (df_2['change_sign'] == 0).sum()
unknown = df_2['change_sign'].isna().sum()

print("Pairwise comparisons (ID, pre, post, sign):")
print(df_2[['ID','pretraining','posttraining','change_sign']].to_string(index=False))
print(f"\nCounts => improvements: {improve}, declines: {decline}, ties: {ties}, unknown: {unknown}")

# Exact binomial test on improvements vs improvements+declines
n = improve + decline
if n == 0:
    print("No comparable pairs (no improvements/declines) to run binomial test.")
else:
    bt = stats.binomtest(improve, n=n, p=0.5)  # two-sided by default
    print(f"Binomial test on {improve}/{n}: p-value = {bt.pvalue:.4f}")
    print("Interpretation: tests if the chance of improvement is different from 0.5")

# why n = improve + decline and not improve + decline + ties
# Because the sign test only tests whether directional changes (improvement vs decline) are equally likely.
# A tie (post == pre) does not give information about direction.
# It does not support the hypothesis of improvement nor the hypothesis of decline.
# Therefore, ties:
# Do not count as evidence for improvement
# Do not count as evidence for decline
# Do not change the null hypothesis
# Are excluded from the binomial denominator
# This is standard for the sign test.

In [ ]:
# tobit_univariate.py
import numpy as np
import pandas as pd
from scipy import optimize, stats, special


df = df_2.copy()

# Censoring threshold: NaN in data means > THRESH
THRESH = 3.0

# Prepare vectors
y = df['posttraining']            # NaN => censored (>THRESH)
y_filled = y.fillna(THRESH)      # for convenience in likelihood
x_pre = df['pretraining']        # NaN means predictor is censored (>THRESH); we'll use fillna(0) trick below

# Tobit negative log-likelihood (right-censoring at THRESH)
# params = [beta0, beta1, log_sigma]
def tobit_negloglike(params, x_pre, y_filled, post_censored_mask, thresh=THRESH):
    beta0, beta1, log_sigma = params
    sigma = np.exp(log_sigma)
    # linear predictor: use intercept + beta1 * pre (for missing pre we use 0 to rely on intercept only)
    xb = beta0 + beta1 * x_pre.fillna(0.0)
    ll = 0.0
    # uncensored observations: log pdf of residuals
    obs_mask = ~post_censored_mask
    if obs_mask.sum() > 0:
        resid = y_filled[obs_mask] - xb[obs_mask]
        ll += stats.norm.logpdf(resid, loc=0, scale=sigma).sum()
    # censored observations: log(1 - Phi((thresh - xb)/sigma)) = log(Phi(-(thresh - xb)/sigma))
    cen_mask = post_censored_mask
    if cen_mask.sum() > 0:
        z = (thresh - xb[cen_mask]) / sigma
        ll += special.log_ndtr(-z).sum()
    return -ll  # negative log-likelihood for minimizer

# initial guess
init_beta0 = np.nanmean(y_filled)
init_beta1 = 0.0
init_logsigma = np.log(np.nanstd(y_filled - init_beta0) + 1e-6)
init_params = np.array([init_beta0, init_beta1, init_logsigma])

post_censored_mask = df['posttraining'].isna()

res = optimize.minimize(
    tobit_negloglike,
    init_params,
    args=(x_pre, y_filled, post_censored_mask, THRESH),
    method='L-BFGS-B'
)

if not res.success:
    print("Optimization warning:", res.message)

beta0_hat, beta1_hat, log_sigma_hat = res.x
sigma_hat = np.exp(log_sigma_hat)

print("Tobit (univariate on posttraining) results:")
print(f"beta0 (intercept): {beta0_hat:.4f}")
print(f"beta1 (pretraining): {beta1_hat:.4f}")
print(f"sigma: {sigma_hat:.4f}")

# Simple (crude) numeric SEs via finite-difference Hessian (optional)
def approx_hessian(f, x0, eps=1e-5):
    n = len(x0)
    hess = np.zeros((n, n))
    fx = f(x0)
    for i in range(n):
        x1 = x0.copy(); x1[i] += eps; f1 = f(x1)
        for j in range(i, n):
            x2 = x0.copy(); x2[j] += eps; f2 = f(x2)
            x3 = x1.copy(); x3[j] += eps; f3 = f(x3)
            hess_ij = (f3 - f1 - f2 + fx) / (eps**2)
            hess[i, j] = hess_ij
            hess[j, i] = hess_ij
    return hess

try:
    hess = approx_hessian(lambda p: tobit_negloglike(p, x_pre, y_filled, post_censored_mask, THRESH), res.x, eps=1e-5)
    cov = np.linalg.inv(hess)
    se = np.sqrt(np.abs(np.diag(cov)))
    print("Approx SEs (beta0, beta1, log_sigma):", se)
except Exception as e:
    print("Could not compute SEs (Hessian invertible?):", e)

# Note: For inference on paired improvement you may prefer a proper paired/bivariate Tobit.


In [ ]:
# wilcoxon_conservative.py
import numpy as np
import pandas as pd
from scipy import stats

THRESH = 3.0  # NaN means value > THRESH

def wilcoxon_with_threshold(df, pre_col='pretraining', post_col='posttraining', thresh=THRESH, alternative='two-sided'):
    # Copy so we do not overwrite original
    df2 = df.copy()
    df2[pre_col] = df2[pre_col].fillna(thresh)   # treat NaN as threshold (conservative)
    df2[post_col] = df2[post_col].fillna(thresh) # same for post if any

    # Keep only pairs where both are not NaN after imputation (they won't be NaN now)
    pre = df2[pre_col].values
    post = df2[post_col].values

    # SciPy's wilcoxon expects differences non-zero? It handles zero-differences (they are treated as ties)
    stat, pvalue = stats.wilcoxon(pre, post, zero_method='wilcox', alternative=alternative)
    return stat, pvalue

# Usage:
stat, p = wilcoxon_with_threshold(df_2)
print("Wilcoxon statistic:", stat, "p-value:", p)

# This is conservative: if an actual true value > THRESH, replacing it by THRESH shrinks the difference (reduces power).
# Use when you want a safe, lower-bound inference that does not invent large values above the threshold.

In [ ]:
# wilcoxon_montecarlo.py
import numpy as np
import pandas as pd
from scipy import stats

# df should already exist in your environment
THRESH = 3.0

def sample_censored(n, thresh, method='shifted_exponential', scale=1.0, delta=1.0, rng=None):
    """
    Draw `n` samples for censored values (>thresh).
    method:
      - 'uniform': Uniform(thresh, thresh + delta)
      - 'shifted_exponential': thresh + Exponential(scale=scale)
    rng: numpy.RandomState or None
    """
    if rng is None:
        rng = np.random.default_rng()

    if method == 'uniform':
        return rng.uniform(low=thresh, high=thresh + delta, size=n)
    elif method == 'shifted_exponential':
        return thresh + rng.exponential(scale=scale, size=n)
    else:
        raise ValueError("Unknown method")

def montecarlo_wilcoxon(df, pre_col='pretraining', post_col='posttraining',
                        thresh=THRESH, sampler_kwargs=None, n_imputations=2000,
                        alternative='two-sided', random_seed=None):
    """
    Monte Carlo multiple-imputation Wilcoxon:
      - For each imputation: draw values for censored pre/post > thresh
      - Run Wilcoxon on the imputed paired data
      - Return distribution of statistics and p-values
    Parameters:
      - sampler_kwargs: dict passed to sample_censored (method, scale, delta)
    """
    rng = np.random.default_rng(random_seed)
    sampler_kwargs = sampler_kwargs or {'method':'shifted_exponential', 'scale':1.0, 'delta':1.0}
    stats_list = []
    pvals = []
    n = len(df)

    # Pre-identify which entries are censored
    pre_censored_idx = df[pre_col].isna().to_numpy()
    post_censored_idx = df[post_col].isna().to_numpy()

    # Observed arrays for non-censored entries
    pre_obs = df[pre_col].to_numpy()
    post_obs = df[post_col].to_numpy()

    for i in range(n_imputations):
        # Start with observed values
        pre_imp = pre_obs.copy().astype(float)
        post_imp = post_obs.copy().astype(float)

        # Impute pre censored entries
        m_pre = pre_censored_idx.sum()
        if m_pre > 0:
            draws_pre = sample_censored(m_pre, thresh, rng=rng, **{k:v for k,v in sampler_kwargs.items() if k in ['method','scale','delta']})
            pre_imp[pre_censored_idx] = draws_pre

        # Impute post censored entries
        m_post = post_censored_idx.sum()
        if m_post > 0:
            draws_post = sample_censored(m_post, thresh, rng=rng, **{k:v for k,v in sampler_kwargs.items() if k in ['method','scale','delta']})
            post_imp[post_censored_idx] = draws_post

        # Now run Wilcoxon on the paired imputed arrays
        try:
            stat, pval = stats.wilcoxon(pre_imp, post_imp, zero_method='wilcox', alternative=alternative)
        except ValueError:
            # This can happen if all differences are zero; handle gracefully
            stat, pval = np.nan, np.nan

        stats_list.append(stat)
        pvals.append(pval)

    stats_arr = np.array(stats_list)
    pvals_arr = np.array(pvals)

    summary = {
        'n_imputations': n_imputations,
        'pvals_median': np.nanmedian(pvals_arr),
        'pvals_mean': np.nanmean(pvals_arr),
        'pvals_proportion_below_0.05': np.nanmean(pvals_arr < 0.05),
        'statistic_median': np.nanmedian(stats_arr),
        'statistic_mean': np.nanmean(stats_arr),
        'pvals': pvals_arr,      # full vector if you want to inspect
        'stats': stats_arr
    }
    return summary

# Usage example:
sampler_kwargs = {'method':'uniform', 'delta':0.5}   # uniform from THRESH..THRESH+0.5
sampler_kwargs = {'method':'shifted_exponential', 'scale':1.0}
summary = montecarlo_wilcoxon(df_2, sampler_kwargs=sampler_kwargs, n_imputations=2000, random_seed=42)
print(summary)
print(summary['pvals_median'], summary['pvals_proportion_below_0.05'])


In [ ]:
# Melt the dataframe
df_melted = df_2.melt(id_vars='ID', value_vars=['pretraining', 'posttraining'],
                    var_name='Stage', value_name='log_arcseconds')
df_melted['Stage'] = df_melted['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})
df_melted['arcseconds'] = 10**df_melted['log_arcseconds'] # to go back in arcseconds
print(10**1.778)
print(10**1.477)

#print(df_melted)

In [ ]:
cols_to_replace = ['pretraining', 'posttraining']
#df = df.dropna()
df_con[cols_to_replace] = df_con[cols_to_replace].replace([3.0], np.nan)
#df_2 = df_2[df_2['pretraining'].notna()]

# Melt the dataframe
df_melted_con = df_con.melt(id_vars='ID', value_vars=['pretraining', 'posttraining'],
                    var_name='Stage', value_name='log_arcseconds')
df_melted_con['Stage'] = df_melted_con['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})
df_melted_con['arcseconds'] = 10**df_melted_con['log_arcseconds'] # to go back in arcseconds
print(df_melted_con)



In [ ]:
# df_melted = df_melted_con.copy() 

# IDs you want to exclude from the *post-training* box stats
outlier_ids = None

# --- 1) Draw the boxplots from a filtered dataframe (stats ignore the 3 post-training points) ---
df_for_boxes = df_melted[
    ~(
        (df_melted['Stage'] == 'Post-training') 
     )
]

### plot manuscript?

### asteroid - log arcsecond

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse

# IDs you want to exclude from the *post-training* box stats
outlier_ids = ['P08', 'P17', 'P19', 'P21', 'P10']

# --- 1) Draw the boxplots from a filtered dataframe (stats ignore the 5 post-training points) ---
df_for_boxes = df_melted[
    ~(
        (df_melted['Stage'] == 'Post-training') &
        (df_melted['ID'].isin(outlier_ids))
     )
]

plt.figure(figsize=(4, 6))
sns.boxplot(
    data=df_for_boxes,
    x='Stage',
    y='log_arcseconds',
    # y = 'arcseconds',
    order=['Pre-Training', 'Post-training'],   # lock the positions: 0 = Pre, 1 = Post
    palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'}, width=0.5
)

plt.axhline(y=2.6, color='blue', zorder=1, linestyle='--')

plt.axhline(y=1.778, color='pink', zorder=1)
plt.axhline(y=1.477, color='pink', zorder=1)
plt.axhspan(1.477, 1.778, color='pink', alpha=0.15, zorder=0)

# plt.axhline(y=60, color='pink', zorder=1)
# plt.axhline(y=30, color='pink', zorder=1)
# plt.axhspan(60, 30, color='pink', alpha=0.15, zorder=0)

# --- 2) Overlay subject labels, highlights, and connecting lines from the *full* dataframe ---
grouped = df_melted.groupby('ID')
jitter = 0.25

# Optional: make jitter reproducible
rng = np.random.default_rng(42)

for ID, group in grouped:
    x_jittered = {}

    for _, row in group.iterrows():
        x_pos = 0 if row['Stage'] == 'Pre-Training' else 1
        jittered_x = x_pos + rng.uniform(-jitter, jitter)
        x_jittered[row['Stage']] = jittered_x

        # If this is one of the 3 special IDs on the post-training side, draw a filled ellipse
        if (row['Stage'] == 'Post-training') and (ID in outlier_ids):
            circle = Ellipse(
                (jittered_x, row['log_arcseconds']),
                width=0.22, height=0.13,
                clip_on = False,
                edgecolor="#A783C9FF", facecolor="#A783C9FF",
                linewidth=0.5, zorder=3
            )
            plt.gca().add_patch(circle)

        # Label each point (draw on top)
        plt.text(
            jittered_x, row['log_arcseconds'],str(ID),
            ha='center', 
            va='center', 
            zorder=5, fontsize=12
        )


    # Connect pre ↔ post if both exist
    if len(group) == 2 and 'Pre-Training' in x_jittered and 'Post-training' in x_jittered:
        y_pre = group[group['Stage'] == 'Pre-Training']['log_arcseconds'].values[0]
        y_post = group[group['Stage'] == 'Post-training']['log_arcseconds'].values[0]

        # Shorten the line a bit by nudging x toward the center
        x_pre = x_jittered['Pre-Training'] + 0.12
        x_post = x_jittered['Post-training'] - 0.12

        plt.plot(
            [x_pre, x_post],
            [y_pre, y_post],
            color='black', alpha=0.3, linewidth=1, zorder=2
        )

plt.text(0.5, 1.51, 'Normal Range', color='red', alpha=0.5, style = 'italic', horizontalalignment='center', verticalalignment='center')

# Thicker tick marks
plt.tick_params(axis='both', width=2, length=6)

# plt.text(0.5, 3, 'Clinical Threshold', color='blue', alpha=0.5, style = 'italic', horizontalalignment='center', verticalalignment='center')

# Example significance bar (adjust y to your data range)
x1, x2 = 0, 1
y, h, col = 3.05, 0.02, 'black'
plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
plt.text((x1+x2)*.5, y+h+0.02, "*", ha='center', va='bottom', color=col)

legend_handles = [
    Line2D([0], [0], color='#D55E00', lw=3, label='  Pre-Training'),
    Line2D([0], [0], color='#009E73', lw=3, label='  Post-Training'),
    Line2D([], [], marker='o', linestyle='none',
           markerfacecolor='#A783C9FF', markeredgecolor='black',
           markersize=15, label='  Post-Training - Gained Stereopsis'),
    Line2D([0], [0], color='blue', linestyle='--', lw=2.5, label='  Clinical Limit')
]

plt.legend(
    handles=legend_handles,
    loc='lower center',
    frameon=True,
    fontsize=10,
    handlelength=3
)

# Final formatting
# plt.title('Stereoacuity', fontsize=18)
plt.ylim(1, 3.2)
plt.xlim(-0.5, 1.5)

plt.ylabel('log Arcseconds', fontsize = 14)
plt.xlabel('')
plt.title('Stereoacuity Threshold', fontsize = 15)
# plt.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.xticks([0, 1], ['Pre-Training', 'Post-Training'], fontsize = 12)  # matches 'order=' above
plt.tight_layout()
# plt.show()
plt.savefig('../Data/Plots/new_new/stereoacuity.png', dpi=600, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(3, 6))

# 1) boxplot (filtered for stats)
sns.boxplot(
    data=df_for_boxes,
    x='Stage',
    y='log_arcseconds',
    order=['Pre-Training', 'Post-training'],
    palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'},
    width=0.5,
    ax=ax,
    zorder=1
)

# 2) prepare deterministic per-subject jitter and x coordinates
stage_to_x = {'Pre-Training': 0.0, 'Post-training': 1.0}
palette = {'Post-training': '#009E73', 'Pre-Training': '#D55E00'}

rng = np.random.default_rng(42)
subject_jitter = {sid: rng.uniform(-0.12, 0.12) for sid in df_melted['ID'].unique()}

# add numeric x and x_plot (numeric + jitter) to dataframe (non-destructive: make a copy first if needed)
df_pts = df_melted.copy()
df_pts['x_numeric'] = df_pts['Stage'].map(stage_to_x)
df_pts['x_plot'] = df_pts.apply(lambda r: r['x_numeric'] + subject_jitter[r['ID']], axis=1)
df_pts['color'] = df_pts['Stage'].map(palette)

# 3) draw the points with scatter (these will be on top of the boxplot)
ax.scatter(
    df_pts['x_plot'],
    df_pts['log_arcseconds'],
    c=df_pts['color'].tolist(),
    edgecolors='black',
    linewidths=0.8,
    s=48,           # point size; tweak as you like
    zorder=3,
    alpha=0.95
)

# 4) draw connecting lines using the SAME jittered x_plot values
for sid, g in df_pts.groupby('ID'):
    # require both stages present
    if {'Pre-Training', 'Post-training'} <= set(g['Stage']):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]
        x_vals = [pre['x_plot'], post['x_plot']]
        y_vals = [pre['log_arcseconds'], post['log_arcseconds']]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1, zorder=2)

# 5) reference lines & formatting (same as before)
ax.axhline(y=2.6, color='blue', zorder=0)
ax.axhline(y=1.778, color='pink', zorder=0)
ax.axhline(y=1.477, color='pink', zorder=0)
ax.axhspan(1.477, 1.778, color='pink', alpha=0.15, zorder=0)

ax.text(0.5, 1.55, 'Normal Range', color='red', alpha=0.5, style='italic',
        ha='center', va='center')

# ax.set_title('Stereoacuity', fontsize=18)
ax.set_ylabel('log arcseconds')
ax.set_xlabel('')
ax.set_ylim(1.25, 3.2)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Pre-Training', 'Post-training'])
ax.grid(True, axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
# plt.show()
plt.savefig('../Data/Plots/Narrative_plots/stereoacuity_clean.png', dpi=600, bbox_inches='tight')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# Convert to arcseconds
# -------------------------
df_pts = df_melted.copy()
df_pts['arcseconds'] = 10 ** df_pts['log_arcseconds']

df_for_boxes_arc = df_for_boxes.copy()
df_for_boxes_arc['arcseconds'] = 10 ** df_for_boxes_arc['log_arcseconds']

# -------------------------
# Figure
# -------------------------
fig, ax = plt.subplots(figsize=(3, 6))

# 1) boxplot (stats computed on filtered data, now in arcseconds)
sns.boxplot(
    data=df_for_boxes_arc,
    x='Stage',
    y='arcseconds',
    order=['Pre-Training', 'Post-training'],
    palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'},
    width=0.5,
    ax=ax,
    zorder=1
)

# -------------------------
# 2) deterministic per-subject jitter (same logic as before)
# -------------------------
stage_to_x = {'Pre-Training': 0.0, 'Post-training': 1.0}
palette = {'Post-training': '#009E73', 'Pre-Training': '#D55E00'}

rng = np.random.default_rng(42)
subject_jitter = {sid: rng.uniform(-0.12, 0.12) for sid in df_pts['ID'].unique()}

df_pts['x_numeric'] = df_pts['Stage'].map(stage_to_x)
df_pts['x_plot'] = df_pts.apply(
    lambda r: r['x_numeric'] + subject_jitter[r['ID']], axis=1
)
df_pts['color'] = df_pts['Stage'].map(palette)

# -------------------------
# 3) scatter points (arcseconds)
# -------------------------
ax.scatter(
    df_pts['x_plot'],
    df_pts['arcseconds'],
    c=df_pts['color'].tolist(),
    edgecolors='black',
    linewidths=0.8,
    s=48,
    zorder=3,
    alpha=0.95
)

# -------------------------
# 4) paired lines (same jittered x)
# -------------------------
for sid, g in df_pts.groupby('ID'):
    if {'Pre-Training', 'Post-training'} <= set(g['Stage']):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]
        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['arcseconds'], post['arcseconds']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=2
        )

# -------------------------
# 5) reference lines (converted from log → arcseconds)
# -------------------------
# original log values
clinical_log = 2.6
normal_low_log = 1.477
normal_high_log = 1.778

clinical_val = 10 ** clinical_log        # ≈ 398 arcsec
normal_low_val = 10 ** normal_low_log    # ≈ 30 arcsec
normal_high_val = 10 ** normal_high_log  # ≈ 60 arcsec

ax.axhline(y=clinical_val, color='blue', zorder=0)
ax.axhline(y=normal_high_val, color='pink', zorder=0)
ax.axhline(y=normal_low_val, color='pink', zorder=0)
ax.axhspan(normal_low_val, normal_high_val, color='pink', alpha=0.15, zorder=0)

ax.text(
    0.5,
    np.sqrt(normal_low_val * normal_high_val),  # geometric midpoint for log scale
    'Normal Range',
    color='red',
    alpha=0.5,
    style='italic',
    ha='center',
    va='center'
)

# -------------------------
# Axes formatting
# -------------------------
ax.set_ylabel('arcseconds')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Pre-Training', 'Post-training'])

# log scale is essential after exponentiation
ax.set_yscale('log')

# match original limits: 10**1.25 → 10**3.2
ax.set_ylim(10 ** 1.25, 10 ** 3.2)

# human-friendly y ticks
yticks = [10, 20, 30, 60, 100, 300, 1000]
ax.set_yticks(yticks)
ax.set_yticklabels([str(int(y)) for y in yticks])

ax.grid(True, axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(
    '../Data/Plots/Narrative_plots/stereoacuity_arcseconds.png',
    dpi=600,
    bbox_inches='tight'
)
# plt.show()


## Crowded Tumbling E
Visual acuity for a Tumbling-E optotype surrounded by flanking contours (crowding),
which amblyopic vision is typically more impaired by than uncrowded acuity.

In [ ]:
df = bv_diff['crowded'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') ].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logMAR')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})

#for the control
df_con = bv_diff_con['crowded'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_con_long = df_con.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logMAR')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)

In [ ]:
df_long_clean = df_long_total.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)


In [ ]:
print(df_long_clean)

In [ ]:


import pingouin as pg 

aov = pg.rm_anova(dv='logMAR', within=['amb','Stage'], subject='ID', data=df_long_clean[(df_long_clean['group'] != 'test')])
aov.round(3)

In [ ]:


import pingouin as pg 

aov = pg.rm_anova(dv='logMAR', within=['amb','Stage'], subject='ID', data=df_long_clean[(df_long_clean['group'] != 'control')])
aov.round(3)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ---------- Input df ----------
# expects df with columns: ID, amb, group ('test'/'control'), Stage ('Pre-Training'/'Post-training'), logMAR
df_vis = df_long_clean.copy()
df_vis['ID'] = df_vis['ID'].astype(str)
df_vis['Stage'] = df_vis['Stage'].astype(str)

# ---------- Split test / control ----------
df_test = df_vis[df_vis['group'].str.lower() == 'test'].copy().reset_index(drop=True)
df_control = df_vis[df_vis['group'].str.lower() == 'control'].copy().reset_index(drop=True)

# ensure amb ordering (use categorical if not already)
if not pd.api.types.is_categorical_dtype(df_vis['amb']):
    amb_order = sorted(df_vis['amb'].unique(), key=str)
    df_vis['amb'] = pd.Categorical(df_vis['amb'], categories=amb_order, ordered=True)
    df_test['amb'] = pd.Categorical(df_test['amb'], categories=amb_order, ordered=True)
    df_control['amb'] = pd.Categorical(df_control['amb'], categories=amb_order, ordered=True)
else:
    amb_order = list(df_vis['amb'].cat.categories)

stage_order = ['Pre-Training', 'Post-training']

# ---------- Plot style ----------
sns.set_theme(style="white")
test_palette = {'Pre-Training': '#D55E00', 'Post-training': '#009E73'}
control_color = '#1E90FF'

# ---------- Compute control ranges per amb × Stage ----------
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'])['logMAR']
    .agg(q1=lambda x: x.quantile(0.25),
         median='median',
         q3=lambda x: x.quantile(0.75),
         minv='min',
         maxv='max')
    .reset_index()
)

# ---------- Figure ----------
fig, ax = plt.subplots(figsize=(7, 6))

box_width = 0.6
range_width = 0.36
offsets = {'Pre-Training': -box_width / 4, 'Post-training': box_width / 4}

# # Draw control shaded ranges (behind)
# for _, row in ctrl_stats.iterrows():
#     amb = row['amb']
#     stage = row['Stage']
#     x_center = amb_order.index(amb) + offsets[stage]

#     # min–max band
#     ax.add_patch(Rectangle(
#         (x_center - range_width / 2, row['minv']),
#         range_width,
#         row['maxv'] - row['minv'],
#         color=control_color,
#         alpha=0.10,
#         zorder=0
#     ))

#     # IQR
#     ax.add_patch(Rectangle(
#         (x_center - range_width / 2, row['q1']),
#         range_width,
#         row['q3'] - row['q1'],
#         color=control_color,
#         alpha=0.18,
#         zorder=0.5
#     ))

#     # median
#     ax.hlines(
#         row['median'],
#         x_center - range_width / 2,
#         x_center + range_width / 2,
#         color=control_color,
#         linewidth=3,
#         zorder=1
#     )

# ---------- Test boxplots ----------
sns.boxplot(
    data=df_test,
    x='amb',
    y='logMAR',
    hue='Stage',
    order=amb_order,
    hue_order=stage_order,
    width=box_width,
    palette=test_palette,
    linewidth=1.5,
    showfliers=False,
    ax=ax,
    zorder=2
)

# ---------- Manual test dots (deterministic jitter) ----------
jitter = 0.06
rng = np.random.default_rng(42)
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['amb'].map({a: i for i, a in enumerate(amb_order)}).astype(float)
df_test['x_offset'] = df_test['Stage'].map(offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['logMAR'],
        color=test_palette[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=4,
        label=stage
    )

# ---------- Connect pre ↔ post per test subject & amb ----------
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]
        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['logMAR'], post['logMAR']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=3
        )

# ---------- Control overall median horizontal line ----------
control_median_all = df_control['logMAR'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)

# ---------- Axes / Legend / Save ----------
ax.set_ylabel("logMAR")
ax.set_xlabel("")
# y-limits and ticks: choose sensible padding from data
ymin = float(df_vis['logMAR'].min())
ymax = float(df_vis['logMAR'].max())
yrange = ymax - ymin if ymax != ymin else 0.5
pad = max(0.1 * yrange, 0.1)

# ax.set_ylim(ymin - pad*0.15, ymax + pad*0.2)
ax.set_ylim(-0.05, 1.9)
ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)

# Customize axis labels
ax.set_ylabel("log of Maximum Angle of Resolution (logMAR)", fontsize = 14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks
ax.set_title('Pre- and Post-Training Crowded Acuity', fontsize = 15)
plt.xticks([0, 1], ['Strong Eye', 'Amblyopic Eye'], fontsize = 13)  # matches 'order=' above


# example significance annotation (edit positions/labels)
x1, x2 = 0, 1
y, h, col = 1.5 - pad*0.05, pad*0.05, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.01, "***", ha='center', va='bottom', color=col, fontsize = 15)

legend_handles = [
    # --- Stage section ---
    # Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color='#D55E00', lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color='#009E73', lw=2.5, label='  Post-Training'),
    Line2D([0], [0], color='#1E90FF', alpha = 0.35, lw=2.5, label='  Control')
]

ax.legend(
    handles=legend_handles,
    loc='upper left',
    frameon=True,
    fontsize=13,
    handlelength=3
)

# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)
    
plt.tight_layout()
plt.savefig('../Data/Plots/new_new/logMAR_crowded.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ---------- Input df ----------
# expects df with columns: ID, amb, group ('test'/'control'), Stage ('Pre-Training'/'Post-training'), logMAR
df_vis = df_long_clean_BE.copy()
df_vis['ID'] = df_vis['ID'].astype(str)
df_vis['Stage'] = df_vis['Stage'].astype(str)

# ---------- Split test / control ----------
df_test = df_vis[df_vis['group'].str.lower() == 'test'].copy().reset_index(drop=True)
df_control = df_vis[df_vis['group'].str.lower() == 'control'].copy().reset_index(drop=True)

# ensure amb ordering (use categorical if not already)
if not pd.api.types.is_categorical_dtype(df_vis['amb']):
    amb_order = sorted(df_vis['amb'].unique(), key=str)
    df_vis['amb'] = pd.Categorical(df_vis['amb'], categories=amb_order, ordered=True)
    df_test['amb'] = pd.Categorical(df_test['amb'], categories=amb_order, ordered=True)
    df_control['amb'] = pd.Categorical(df_control['amb'], categories=amb_order, ordered=True)
else:
    amb_order = list(df_vis['amb'].cat.categories)

stage_order = ['Pre-Training', 'Post-training']

# ---------- Plot style ----------
sns.set_theme(style="white")
test_palette = {'Pre-Training': '#D55E00', 'Post-training': '#009E73'}
control_color = '#1E90FF'

# ---------- Compute control ranges per amb × Stage ----------
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'])['logMAR']
    .agg(q1=lambda x: x.quantile(0.25),
         median='median',
         q3=lambda x: x.quantile(0.75),
         minv='min',
         maxv='max')
    .reset_index()
)

# ---------- Figure ----------
fig, ax = plt.subplots(figsize=(7, 6))

box_width = 0.6
range_width = 0.36
offsets = {'Pre-Training': -box_width / 4, 'Post-training': box_width / 4}

# Draw control shaded ranges (behind)
for _, row in ctrl_stats.iterrows():
    amb = row['amb']
    stage = row['Stage']
    x_center = amb_order.index(amb) + offsets[stage]

    # min–max band
    ax.add_patch(Rectangle(
        (x_center - range_width / 2, row['minv']),
        range_width,
        row['maxv'] - row['minv'],
        color=control_color,
        alpha=0.10,
        zorder=0
    ))

    # IQR
    ax.add_patch(Rectangle(
        (x_center - range_width / 2, row['q1']),
        range_width,
        row['q3'] - row['q1'],
        color=control_color,
        alpha=0.18,
        zorder=0.5
    ))

    # median
    ax.hlines(
        row['median'],
        x_center - range_width / 2,
        x_center + range_width / 2,
        color=control_color,
        linewidth=3,
        zorder=1
    )

# ---------- Test boxplots ----------
sns.boxplot(
    data=df_test,
    x='amb',
    y='logMAR',
    hue='Stage',
    order=amb_order,
    hue_order=stage_order,
    width=box_width,
    palette=test_palette,
    linewidth=1.5,
    showfliers=False,
    ax=ax,
    zorder=2
)

# ---------- Manual test dots (deterministic jitter) ----------
jitter = 0.06
rng = np.random.default_rng(42)
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['amb'].map({a: i for i, a in enumerate(amb_order)}).astype(float)
df_test['x_offset'] = df_test['Stage'].map(offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['logMAR'],
        color=test_palette[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=4,
        label=stage
    )

# ---------- Connect pre ↔ post per test subject & amb ----------
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-training'].iloc[0]
        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['logMAR'], post['logMAR']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=3
        )

# ---------- Control overall median horizontal line ----------
control_median_all = df_control['logMAR'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)

# ---------- Axes / Legend / Save ----------
ax.set_ylabel("logMAR")
ax.set_xlabel("")
# y-limits and ticks: choose sensible padding from data
ymin = float(df_vis['logMAR'].min())
ymax = float(df_vis['logMAR'].max())
yrange = ymax - ymin if ymax != ymin else 0.5
pad = max(0.1 * yrange, 0.1)
ax.set_ylim(ymin - pad*0.15, ymax + pad*0.2)

ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)

# example significance annotation (edit positions/labels)
x1, x2 = 0, 1
y, h, col = ax.get_ylim()[1] - pad*0.05, pad*0.03, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.01, "***", ha='center', va='bottom', color=col)

# clean legend: keep one entry per Stage
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))
ax.legend(
    [uniq[k] for k in stage_order if k in uniq],
    [k for k in stage_order if k in uniq],
    loc='upper left',
    handlelength=3
)

plt.tight_layout()
plt.savefig('../Data/Plots/Narrative_plots/logMAR_cutoff_style.png', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:

# Set a better style
sns.set_theme(style="white", palette="pastel")

# Set up the plot
plt.figure(figsize=(4, 6))

# Create the boxplot with colors for each amb condition
ax = sns.boxplot(
    x='amb', y='logMAR', data=df_long_clean_BE, hue='Stage', showfliers=False, linewidth=1.5, 
    #palette={'Lazy Eye': '#66CDAA', 'Both Eyes': '#1E90FF', 'Strong Eye': '#FF6347'}
    palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'}
)

ax.axhline(y=22.1, color='blue', linestyle='--', linewidth=3, zorder=1, alpha=0.2)

strip = sns.stripplot(data=df_long_clean_BE, x="amb", y="logMAR", hue='Stage', dodge=True, ax=ax, 
                palette={'Post-training': '#009E73', 'Pre-Training': '#D55E00'}, edgecolor='black', linewidth=1)

# ax.set_yticks(range(0, 35, 5))
ax.set_ylim(0, 1)
#ax.set_yticklabels([5, 10, 15, 20, 25])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[(handles[0]), (handles[1])],
          labels=['Pre-Training', 'Post-training'],
          loc='upper left', handlelength=3,
          handler_map={tuple: HandlerTuple(ndivide=None)})

# Map amb categories to numerical positions for plotting
amb_order = df['amb'].unique()
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# Customize axis labels
ax.set_ylabel("log Maximum Angle of Resolution (logMAR)")  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks

# Get cleaned categorical order for amb
amb_order = df_long_clean_BE['amb'].cat.categories
amb_mapping = {val: i for i, val in enumerate(amb_order)}

# === Extract real x/y points from stripplot ===
# Make sure the order of the DataFrame matches how seaborn plots the points
# Seaborn plots in order: hue nested inside x, so sort by 'amb' then 'Stage'
df_sorted = df_long_clean_BE.sort_values(by=['amb', 'Stage'], key=lambda col: col.astype(str))

# Get all stripplot points
offsets = []
for coll in strip.collections:
    offsets.extend(coll.get_offsets())

# Map points to their (ID, amb) and Stage
coord_map = {}  # {(ID, amb): {stage: (x, y)}}
for i, (_, row) in enumerate(df_sorted.iterrows()):
    x, y = offsets[i]
    key = (row['ID'], row['amb'])
    stage = row['Stage']
    if key not in coord_map:
        coord_map[key] = {}
    coord_map[key][stage] = (x, y)

# Plot lines for each pair of Pre/Post
for key, stages in coord_map.items():
    if 'Pre-Training' in stages and 'Post-training' in stages:
        x_vals = [stages['Pre-Training'][0], stages['Post-training'][0]]
        y_vals = [stages['Pre-Training'][1], stages['Post-training'][1]]
        ax.plot(x_vals, y_vals, color='black', alpha=0.4, linewidth=1)

# # Example of adding a significance bar
# x1, x2 = 0, 1  # x locations (you can use actual x from amb_mapping + hue offsets)
# y, h, col = 0.8, 0.5, 'black'
# # ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
# # ax.text((x1+x2)*.5, y+h+0.2, "n.s.", ha='center', va='bottom', color=col)

# ax.plot([x1-0.15, x1-0.15, x1+0.15, x1+0.15], [1, 1+h, 1+h, 1], lw=1.5, c=col)
# ax.text((x1+x1)*.5, 0.8+h, "n.s.", ha='center', va='bottom', color=col)

#plt.title('No Acuity Improvement for Amblyopes after Training', fontsize = 16)
#plt.figure(figsize=(8, 6))
plt.savefig('../Data/Plots/Narrative_plots/Tumbling_crowded_supplementary.png', dpi=600, bbox_inches='tight')



In [ ]:
print(df_long_clean_BE)

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_clean_BE[(df_long_clean_BE['group'] != 'control')][df_long_clean_BE[(df_long_clean_BE['group'] != 'control')]['Stage'] == 'Pre-Training']['logMAR'].values,
    y=df_long_clean_BE[(df_long_clean_BE['group'] != 'control')][df_long_clean_BE[(df_long_clean_BE['group'] != 'control')]['Stage'] == 'Post-training']['logMAR'].values
)

print(ttest.round(3))

In [ ]:
df_amb = df.copy()
df_amb = df_amb[(df_amb['amb'] == 'Lazy Eye')]

In [ ]:

# Create the figure and axes
plt.figure(figsize=(4, 4))

# Plot each point as a text label (ID)
for i, row in df_amb.iterrows():
    plt.text(row['pretraining'], row['posttraining'], str(row['ID']),
             fontsize=14, ha='center', va='center')

# Add reference line y = x
plt.plot([df_amb['pretraining'].min(), df_amb['pretraining'].max()+0.12],
         [df_amb['pretraining'].min(), df_amb['pretraining'].max()+0.12],
         color='gray', linestyle='--')

#plt.xticks(range(0, 1.2, 0.2))
#plt.yticks(range(0, 1.2, 0.2))
#plt.xlim(0, 1.2)
#plt.ylim(0, 1.2)

# Labels and title
plt.xlabel("Pre-training logMAR")
plt.ylabel("Post-training logMAR")
plt.title("Crowded Acuity (Pre vs Post Training)")
plt.grid(True)
plt.tight_layout()
plt.show()

## Tumbling E
Uncrowded visual acuity for the Tumbling-E optotype (no flanking contours).

In [ ]:

df = bv_diff['snellen'].copy()
df['group'] = 'test'
indexID = df[ (df['ID'] == 'P01') | (df['ID'] == 'P02') | (df['ID'] == 'P03') ].index # | (df['ID'] == 'P02') | (df['ID'] == 'P03')
df.drop(indexID , inplace=True)
df['amb'] = df['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_long = df.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logMAR')
df_long['Stage'] = df_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

#for the control
df_con = bv_diff_con['snellen'].copy()
df_con['group'] = 'control'
df_con['amb'] = df_con['amb'].map({'Amb': 'Amblyopic Eye', 'Both': 'Both Eyes', 'Strong': 'Strong Eye'})
df_con_long = df_con.melt(id_vars=['ID', 'amb', 'group'], value_vars=['pretraining', 'posttraining'], 
                  var_name='Stage', value_name='logMAR')
df_con_long['Stage'] = df_con_long['Stage'].map({'pretraining': 'Pre-Training', 'posttraining': 'Post-Training'})

# merge by stacking rows (use concat, not merge)
df_total = pd.concat([df, df_con], ignore_index=True)
df_long_total = pd.concat([df_long, df_con_long], ignore_index=True)
# print(df_total)
print(df_long_total)

In [ ]:
df_long_clean = df_long_total.copy()
df_long_clean_BE = df_long_clean[(df_long_clean['amb'] == 'Both Eyes')]
df_long_clean_BE['amb'] = pd.Categorical(df_long_clean_BE['amb'], categories=['Both Eyes'], ordered=True)

df_long_clean = df_long_clean[(df_long_clean['amb'] != 'Both Eyes')]
df_long_clean['amb'] = pd.Categorical(df_long_clean['amb'], categories=['Strong Eye', 'Amblyopic Eye'], ordered=True)


In [ ]:

import pingouin as pg 

aov = pg.rm_anova(dv='logMAR', within=['amb','Stage'], subject='ID', data=df_long_clean)
aov.round(3)

In [ ]:
# Run paired t-test
ttest = pg.ttest(
    paired=True,
    x=df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Pre-Training']['logMAR'].values,
    y=df_long_clean[(df_long_clean['group'] != 'control')][df_long_clean['amb'] == 'Amblyopic Eye'][df_long_clean['Stage']== 'Post-Training']['logMAR'].values
)
print(ttest.round(3))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# ---------- Input df ----------
# expects df with columns: ID, amb, group ('test'/'control'), Stage ('Pre-Training'/'Post-training'), logMAR
df_vis = df_long_clean.copy()
df_vis['ID'] = df_vis['ID'].astype(str)
df_vis['Stage'] = df_vis['Stage'].astype(str)

# ---------- Split test / control ----------
df_test = df_vis[df_vis['group'].str.lower() == 'test'].copy().reset_index(drop=True)
df_control = df_vis[df_vis['group'].str.lower() == 'control'].copy().reset_index(drop=True)

# ensure amb ordering (use categorical if not already)
if not pd.api.types.is_categorical_dtype(df_vis['amb']):
    amb_order = sorted(df_vis['amb'].unique(), key=str)
    df_vis['amb'] = pd.Categorical(df_vis['amb'], categories=amb_order, ordered=True)
    df_test['amb'] = pd.Categorical(df_test['amb'], categories=amb_order, ordered=True)
    df_control['amb'] = pd.Categorical(df_control['amb'], categories=amb_order, ordered=True)
else:
    amb_order = list(df_vis['amb'].cat.categories)

stage_order = ['Pre-Training', 'Post-Training']

# ---------- Plot style ----------
sns.set_theme(style="white")
test_palette = {'Pre-Training': '#D55E00', 'Post-Training': '#009E73'}
control_color = '#1E90FF'

# ---------- Compute control ranges per amb × Stage ----------
ctrl_stats = (
    df_control
    .groupby(['amb', 'Stage'])['logMAR']
    .agg(q1=lambda x: x.quantile(0.25),
         median='median',
         q3=lambda x: x.quantile(0.75),
         minv='min',
         maxv='max')
    .reset_index()
)

# ---------- Figure ----------
fig, ax = plt.subplots(figsize=(7, 6))

box_width = 0.6
range_width = 0.36
offsets = {'Pre-Training': -box_width / 4, 'Post-Training': box_width / 4}

# # Draw control shaded ranges (behind)
# for _, row in ctrl_stats.iterrows():
#     amb = row['amb']
#     stage = row['Stage']
#     x_center = amb_order.index(amb) + offsets[stage]

#     # min–max band
#     ax.add_patch(Rectangle(
#         (x_center - range_width / 2, row['minv']),
#         range_width,
#         row['maxv'] - row['minv'],
#         color=control_color,
#         alpha=0.10,
#         zorder=0
#     ))

#     # IQR
#     ax.add_patch(Rectangle(
#         (x_center - range_width / 2, row['q1']),
#         range_width,
#         row['q3'] - row['q1'],
#         color=control_color,
#         alpha=0.18,
#         zorder=0.5
#     ))

#     # median
#     ax.hlines(
#         row['median'],
#         x_center - range_width / 2,
#         x_center + range_width / 2,
#         color=control_color,
#         linewidth=3,
#         zorder=1
#     )

# ---------- Test boxplots ----------
sns.boxplot(
    data=df_test,
    x='amb',
    y='logMAR',
    hue='Stage',
    order=amb_order,
    hue_order=stage_order,
    width=box_width,
    palette=test_palette,
    linewidth=1.5,
    showfliers=False,
    ax=ax,
    zorder=2
)

# ---------- Manual test dots (deterministic jitter) ----------
jitter = 0.06
rng = np.random.default_rng(42)
df_test = df_test.reset_index(drop=True)
df_test['x_base'] = df_test['amb'].map({a: i for i, a in enumerate(amb_order)}).astype(float)
df_test['x_offset'] = df_test['Stage'].map(offsets).astype(float)
df_test['x_jitter'] = rng.uniform(-jitter, jitter, size=len(df_test)).astype(float)
df_test['x_plot'] = df_test['x_base'] + df_test['x_offset'] + df_test['x_jitter']

for stage in stage_order:
    sub = df_test[df_test['Stage'] == stage]
    ax.scatter(
        sub['x_plot'],
        sub['logMAR'],
        color=test_palette[stage],
        edgecolors='black',
        linewidths=0.8,
        s=40,
        alpha=0.95,
        zorder=4,
        label=stage
    )

# ---------- Connect pre ↔ post per test subject & amb ----------
for (pid, amb), g in df_test.groupby(['ID', 'amb']):
    if set(stage_order).issubset(g['Stage'].values):
        pre = g[g['Stage'] == 'Pre-Training'].iloc[0]
        post = g[g['Stage'] == 'Post-Training'].iloc[0]
        ax.plot(
            [pre['x_plot'], post['x_plot']],
            [pre['logMAR'], post['logMAR']],
            color='black',
            alpha=0.4,
            linewidth=1,
            zorder=3
        )

# ---------- Control overall median horizontal line ----------
control_median_all = df_control['logMAR'].median()
ax.axhline(
    y=control_median_all,
    color=control_color,
    linestyle='--',
    linewidth=3,
    alpha=0.35,
    zorder=1
)

# ---------- Axes / Legend / Save ----------
ax.set_ylabel("logMAR")
ax.set_xlabel("")
# y-limits and ticks: choose sensible padding from data
ymin = float(df_vis['logMAR'].min())
ymax = float(df_vis['logMAR'].max())
yrange = ymax - ymin if ymax != ymin else 0.5
pad = max(0.1 * yrange, 0.1)

# ax.set_ylim(ymin - pad*0.15, ymax + pad*0.2)
ax.set_ylim(-0.05, 1.9)
ax.set_xticks(range(len(amb_order)))
ax.set_xticklabels(amb_order)

# Customize axis labels
ax.set_ylabel("log of Maximum Angle of Resolution (logMAR)", fontsize = 14)  # or whatever label you'd like
ax.set_xlabel("")  # removes the label but keeps 'Strong Eye' / 'Lazy Eye' ticks
ax.set_title('Pre- and Post-Training Tumbling E Acuity', fontsize = 15)
plt.xticks([0, 1], ['Strong Eye', 'Amblyopic Eye'], fontsize = 13)  # matches 'order=' above


# example significance annotation (edit positions/labels)
x1, x2 = 0, 1
y, h, col = 1.4 - pad*0.05, pad*0.05, 'black'
ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
ax.text((x1+x2)*.5, y+h+0.01, "***", ha='center', va='bottom', color=col, fontsize = 15)

legend_handles = [
    # --- Stage section ---
    # Line2D([], [], linestyle='none', label='Stage'),
    Line2D([0], [0], color='#D55E00', lw=2.5, label='  Pre-Training'),
    Line2D([0], [0], color='#009E73', lw=2.5, label='  Post-Training'),
    Line2D([0], [0], color='#1E90FF', alpha = 0.35, lw=2.5, label='  Control')
]

ax.legend(
    handles=legend_handles,
    loc='upper left',
    frameon=True,
    fontsize=13,
    handlelength=3
)

# Thicker tick marks
ax.tick_params(axis='both', width=2, length=6)

# Thicker box (spines)
for spine in ax.spines.values():
    spine.set_linewidth(2)
    
plt.tight_layout()
plt.savefig('../Data/Plots/new_new/logMAR_tumbling.png', dpi=600, bbox_inches='tight')
plt.show()
